# U02 SQL（一）：關聯模型、建表約束與單表查詢

**資料庫管理**　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

先用一張文件表，完整操作一次建表與資料增刪改查；每項進階寫法都從這個基本模型增加一個需求。SQL 識別字使用英文，中文用於解說與資料值。

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

# 起手式：一張文件表的完整生命週期

以 `doc-finder.py` 的文件清單為情境，做一個刻意縮小的版本：一列代表一份文件，只保留 `id`、`filename`、`path`。這不是原程式的完整 schema；本例不掃硬碟、不開檔，也不修改真實的 `doc.db`。

這一輪只問五件事：表怎麼建立？資料怎麼新增？怎麼看？怎麼改？怎麼刪？先手動給 ID，不處理自動編號或重複資料。

程式格中，字串裡的是 SQL；`execute()` 是 Python 把 SQL 交給資料庫，`fetchall()` 把查詢結果取回。`commit()` 是確認保存本次修改的 Python 呼叫。本例用記憶體資料庫，確認後仍不會產生磁碟檔案。

## ① CREATE TABLE：定義空表

`INTEGER` 表達整數用途，`TEXT` 表達文字用途；`TEXT` 是 SQLite／PostgreSQL 都有的型別名稱，不是 ANSI 型別名稱。建表只定義欄位，不會產生文件紀錄。

In [ ]:
import sqlite3

# 從本格開始重跑，就會得到新的獨立練習資料庫。
doc_con = sqlite3.connect(":memory:")
doc_con.execute("""
CREATE TABLE doc (
    id INTEGER,
    filename TEXT,
    path TEXT
)
""")
print("已建立 doc 空表：id、filename、path")

## ② INSERT：新增一列

`INSERT INTO doc (欄位清單) VALUES (對應的值)`：左右位置一一對應。文字值用單引號，數字直接寫。現在每句只新增一列，ID 由我們手動填 1、2。

In [ ]:
doc_con.execute("""
INSERT INTO doc (id, filename, path)
VALUES (1, 'sql-notes.pdf', '/docs/sql-notes.pdf')
""")
doc_con.execute("""
INSERT INTO doc (id, filename, path)
VALUES (2, 'python-notes.pdf', '/docs/python-notes.pdf')
""")
doc_con.commit()
print("新增了兩筆文件紀錄；不會建立兩個 PDF 檔案。")

## ③ SELECT：看看目前有哪些資料

`SELECT` 指定要看的欄位，`FROM` 指定表；`ORDER BY id` 讓顯示依 ID 排序，沒有它不保證列的順序。加上 `WHERE id = 1` 就只挑符合條件的列。查詢不會修改原表。

In [ ]:
print("全部文件：")
print(doc_con.execute("SELECT id, filename, path FROM doc ORDER BY id").fetchall())
print("只看 id = 1：")
print(doc_con.execute("SELECT id, filename, path FROM doc WHERE id = 1").fetchall())
# 預期：全部有 id 1、2；第二次只看到 id 1。

## ④ UPDATE：修改既有列

`UPDATE doc SET filename = 新值 WHERE id = 1`：WHERE 決定改誰，SET 決定改哪個欄位及新值。這不是新增一列；沒有指定的欄位保留原值。

忘記 WHERE 就會修改整張表。先用 SELECT 預覽相同條件，是檢查自己是否挑錯資料的基本動作；在多人同時操作的系統中，預覽本身不保證資料不被別人改動。

In [ ]:
print("修改前：", doc_con.execute(
    "SELECT id, filename, path FROM doc WHERE id = 1"
).fetchall())
doc_con.execute("UPDATE doc SET filename = 'sql-intro.pdf' WHERE id = 1")
doc_con.commit()
print("修改後：", doc_con.execute(
    "SELECT id, filename, path FROM doc ORDER BY id"
).fetchall())
# 仍有兩列，只有 id 1 的 filename 改變；path 沒有改，硬碟檔案也不會被重新命名。

## ⑤ DELETE：刪除符合條件的列

`DELETE FROM doc WHERE id = 2` 刪的是整列，不是把某個欄位清空；WHERE 仍然決定刪誰。忘記 WHERE 會刪掉全表的列，但表本身仍存在。

In [ ]:
print("準備刪除：", doc_con.execute(
    "SELECT id, filename, path FROM doc WHERE id = 2"
).fetchall())
doc_con.execute("DELETE FROM doc WHERE id = 2")
doc_con.commit()
doc_remaining = doc_con.execute("SELECT id, filename, path FROM doc ORDER BY id").fetchall()
print("刪除後：", doc_remaining)
assert doc_remaining == [(1, 'sql-intro.pdf', '/docs/sql-notes.pdf')]
# 刪除的是資料庫紀錄，不會刪除硬碟上的 PDF。

## 第一層加法：命令不變，先把資料規則寫清楚

完整生命週期已經跑過：**空表 → 兩列 → 修改其中一列 → 刪除其中一列**。CRUD 指資料的 Create／Read／Update／Delete，其中資料的 Create 對應 INSERT；CREATE TABLE 則是建立容器。

現在才問：ID 重複、檔名沒填、同一路徑登記兩次怎麼辦？第一版的普通欄位不會擋住這些情況。建立一張加規則的 `doc_checked` 對照表：

- `PRIMARY KEY`：識別一列，ID 不能重複；本例仍手動給值。
- `NOT NULL`：不接受缺值 NULL；它不等於禁止空字串。
- `UNIQUE`：這裡要求已填寫的 path 不重複。

這些是**資料約束，不只是語法 sugar**：它們會改變哪些資料能被接受。CREATE 的欄位上多了規則，INSERT／SELECT／UPDATE／DELETE 的基本骨架沒有改。

In [ ]:
doc_con.execute("""
CREATE TABLE doc_checked (
    id INTEGER PRIMARY KEY,
    filename TEXT NOT NULL,
    path TEXT NOT NULL UNIQUE
)
""")
doc_con.execute("""
INSERT INTO doc_checked (id, filename, path)
VALUES (1, 'sql-intro.pdf', '/docs/sql-notes.pdf')
""")
doc_con.commit()
print(doc_con.execute("SELECT id, filename, path FROM doc_checked ORDER BY id").fetchall())
# 此處只比較宣告與正常寫入；資料規則不需每次 INSERT 都手動重寫。

## 第二層加法：資料來自使用者，SQL 骨架不變

前面的固定字串方便看懂語法；程式真正接收檔名時，使用 `doc-finder.py` 那樣的參數綁定。`?` 是 Python sqlite3 的值佔位符，值用第二個參數傳入，不用 f-string 拼 SQL，也不必自行處理檔名裡的單引號。

這是**Python API 的安全用法，不是新增一種 SQL 命令**。`?` 不能替代表名或欄名；換 PostgreSQL driver 時，佔位符形式要依該 driver 調整。

In [ ]:
doc_filename = "reader's-notes.pdf"
doc_path = "/docs/reader's-notes.pdf"
doc_con.execute(
    "INSERT INTO doc_checked (id, filename, path) VALUES (?, ?, ?)",
    (2, doc_filename, doc_path)
)
doc_con.execute(
    "UPDATE doc_checked SET filename = ? WHERE id = ?",
    ("reader's-sql-notes.pdf", 2)
)
doc_con.commit()
print(doc_con.execute("SELECT id, filename, path FROM doc_checked ORDER BY id").fetchall())
assert doc_con.execute("SELECT filename FROM doc_checked WHERE id = 2").fetchone()[0] == "reader's-sql-notes.pdf"

## 用同一個基本模型理解增加的語法

每次看到新寫法，都問它增加什麼：**拒絕壞資料、補預設值、改變選列條件、計算新值，還是省下重複書寫？**

例如 `INSERT ... VALUES (...), (...)` 是把多列寫在同一句裡；`DEFAULT` 是省略某欄時提供初值；`UPDATE ... SET a = ..., b = ...` 是一次改多欄。它們不是三套全新的生命週期，也不是全部都只改外觀。批次成一句還會影響發生錯誤時的處理範圍。

小練習：在 doc 新增 id = 3 的文件、查出它、改它的 filename、再刪除它。只需套用剛才五種命令，不需其他語法。

## 語法標籤的讀法

本講義以 SQLite 執行。文中的「標準 SQL」指 ANSI／ISO SQL 的功能與基本形式；「共有擴充」指 SQLite 與 PostgreSQL 都有、但不是標準的寫法；「SQLite 用法」則需在換引擎時重新檢查。相同語法仍可能有型別、NULL 與版本差異。SQL 識別字使用英文，中文用於解說與資料值。

In [ ]:
#@title 📦 資料準備：建立課程範例資料庫 univ.db（建立五張表、插入示範資料並定義查詢函數 q）
import sqlite3, os, pandas as pd

if os.path.exists("univ.db"):
    os.remove("univ.db")
con = sqlite3.connect("univ.db")
con.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE student(
  sid   TEXT PRIMARY KEY NOT NULL,          -- 學號
  name  TEXT NOT NULL,             -- 姓名
  dept  TEXT NOT NULL,             -- 系所
  year  INTEGER CHECK(year BETWEEN 1 AND 4)   -- 年級
);
CREATE TABLE instructor(
  iid   TEXT PRIMARY KEY NOT NULL,
  name  TEXT NOT NULL,
  dept  TEXT NOT NULL,
  salary REAL
);
CREATE TABLE course(
  cid     TEXT PRIMARY KEY NOT NULL,
  title   TEXT NOT NULL,
  dept    TEXT NOT NULL,
  credits INTEGER NOT NULL DEFAULT 3
);
CREATE TABLE takes(                -- 修課紀錄
  sid TEXT NOT NULL REFERENCES student(sid),
  cid TEXT NOT NULL REFERENCES course(cid),
  semester TEXT NOT NULL,                   -- 學期，如 114-1
  grade REAL,                      -- NULL = 在修中
  PRIMARY KEY (sid, cid, semester)
);
CREATE TABLE teaches(              -- 授課紀錄
  iid TEXT NOT NULL REFERENCES instructor(iid),
  cid TEXT NOT NULL REFERENCES course(cid),
  semester TEXT NOT NULL,
  PRIMARY KEY (iid, cid, semester)
);
""")
con.executemany("INSERT INTO student VALUES (?,?,?,?)", [
 ("S001","林佳蓉","統計",3),("S002","陳威廷","統計",3),("S003","張雅筑","統計",3),
 ("S004","李承翰","統計",2),("S005","王思穎","統計",4),("S006","黃冠宇","統計",3),
 ("S007","吳孟軒","資訊",3),("S008","劉子涵","資訊",2),("S009","蔡明修","資訊",4),
 ("S010","許芷瑄","數學",3),("S011","鄭宇翔","數學",2),("S012","謝欣妤","數學",4),
 ("S013","洪偉倫","企管",3),("S014","郭品妍","企管",2),("S015","曾柏勳","企管",3),
 ("S016","賴韻如","統計",1),("S017","周家豪","資訊",1),("S018","江美慧","統計",4),
 ("S019","趙國彬","數學",3),("S020","方語彤","企管",4),
])
con.executemany("INSERT INTO instructor VALUES (?,?,?,?)", [
 ("I01","王教授","統計",118000.0),("I02","李教授","統計",102000.0),
 ("I03","張教授","資訊",111000.0),("I04","陳教授","數學",96000.0),
 ("I05","林教授","企管",99000.0),("I06","徐教授","統計",None),
])
con.executemany("INSERT INTO course VALUES (?,?,?,?)", [
 ("C101","統計學（一）","統計",3),("C102","迴歸分析","統計",3),
 ("C103","資料庫管理","統計",3),("C104","機率論","統計",3),
 ("C201","微積分","數學",4),("C202","線性代數","數學",3),
 ("C301","程式設計","資訊",3),("C302","資料結構","資訊",3),
])
con.executemany("INSERT INTO takes VALUES (?,?,?,?)", [
 ("S001","C101","114-1",88),("S001","C104","114-1",92),("S001","C102","114-2",85),
 ("S001","C103","115-1",None),("S002","C101","114-1",76),("S002","C102","114-2",81),
 ("S002","C103","115-1",None),("S003","C101","114-1",95),("S003","C104","114-1",89),
 ("S003","C102","114-2",91),("S003","C103","115-1",None),("S004","C101","114-2",67),
 ("S004","C201","114-2",72),("S004","C104","115-1",None),("S005","C101","113-1",82),
 ("S005","C102","113-2",78),("S005","C103","114-1",90),("S006","C101","114-1",58),
 ("S006","C104","114-1",61),("S006","C103","115-1",None),("S007","C301","114-1",93),
 ("S007","C302","114-2",87),("S007","C103","115-1",None),("S008","C301","114-2",74),
 ("S008","C302","115-1",None),("S009","C301","113-1",85),("S009","C302","113-2",80),
 ("S009","C202","114-1",77),("S010","C201","114-1",90),("S010","C202","114-2",94),
 ("S011","C201","114-2",63),("S011","C202","115-1",None),("S012","C201","113-1",71),
 ("S012","C202","113-2",75),("S012","C101","114-1",79),("S013","C101","114-1",70),
 ("S013","C103","115-1",None),("S014","C101","114-2",55),("S015","C101","114-1",83),
 ("S015","C102","114-2",88),("S016","C101","115-1",None),("S017","C301","115-1",None),
 ("S018","C101","113-1",96),("S018","C102","113-2",93),("S018","C104","114-1",98),
 ("S018","C103","114-1",94),("S019","C201","114-1",84),("S019","C202","114-2",86),
 ("S020","C101","113-2",73),("S020","C102","114-1",69),
])
con.executemany("INSERT INTO teaches VALUES (?,?,?)", [
 ("I01","C101","114-1"),("I01","C101","114-2"),("I01","C104","114-1"),
 ("I02","C102","114-2"),("I02","C102","114-1"),("I06","C103","115-1"),
 ("I06","C103","114-1"),("I03","C301","114-1"),("I03","C302","114-2"),
 ("I04","C201","114-1"),("I04","C202","114-2"),("I05","C101","113-2"),
])
con.commit()

def q(sql, params=()):
    """跑一句 SELECT，回傳 pandas DataFrame（Colab 會漂亮顯示）"""
    return pd.read_sql_query(sql, con, params=params)

for t in ["student","instructor","course","takes","teaches"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<12}{n:>4} 列")
print("SQLite version:", sqlite3.sqlite_version)
print("foreign_keys:", con.execute("PRAGMA foreign_keys").fetchone()[0])
print("univ.db 就緒 ✅")

### univ.db 綱要（這學期的老朋友）

```
student(sid PK, name, dept, year)          instructor(iid PK, name, dept, salary)
course(cid PK, title, dept, credits)       teaches(iid→instructor, cid→course, semester)
takes(sid→student, cid→course, semester, grade)      ← grade 為 NULL 表示在修中
```

「→」是 **foreign key**：`takes.sid` 在此設為必填，其值必須存在於 `student.sid`。查詢前可先確認每張表的一列代表什麼。

# 第 0 節：關聯模型——資料的數學形狀

## 0.1 Codd 的天才提案：所有資料都是「表」

用統計系熟悉的語言：一個 **relation** 就是笛卡兒積的一個有限子集

$$r \subseteq D_1 \times D_2 \times \cdots \times D_n$$

其中 $D_i$ 是第 $i$ 個屬性（attribute／欄位）的值域（domain）。每個元素是一個 **tuple**（列）。

三個立刻要記的心智設定（跟 DataFrame 最大的差異）：

1. **schema vs instance**：表的「格式」vs 某一刻的「內容」。
2. 理論上**列沒有順序、不重複**（集合！）；要順序，查詢時自己 `ORDER BY`。
3. 「relational」指的是 relation（表）這個數學物件，**不是**「表跟表有關係」——常見誤解。

術語：欄數叫 **arity**（度數），列數叫 **cardinality**（基數）。例如三欄、三列的表，arity 與 cardinality 都是 3。

In [ ]:
# relation ＝ 笛卡兒積的有限子集：把定義跑一次
from itertools import product

dom_sid  = {"S001", "S002", "S003"}          # 學號的值域（示意）
dom_dept = {"統計", "資訊"}                   # 系所的值域
dom_year = {3, 4}                             # 年級的值域

universe = set(product(dom_sid, dom_dept, dom_year))            # D1 × D2 × D3：所有「可能」的列
r = {("S001", "統計", 3), ("S002", "統計", 4), ("S003", "資訊", 3)}   # 某一刻的 instance

print(f"|dom_sid × dom_dept × dom_year| = {len(universe)}（3×2×2 種可能）")
print("r ⊆ 笛卡兒積？", r <= universe)
print("arity（欄數）= 3、cardinality（列數）=", len(r))
print("集合沒有順序：",
      {("S001","統計",3), ("S002","統計",4)} == {("S002","統計",4), ("S001","統計",3)})

In [ ]:
# 查詢＝集合運算：選擇 σ（挑列）與投影 π（挑欄）——SQL 的數學前身
sel = {t for t in r if t[1] == "統計"}          # σ_[dept='統計'](r)   → SQL 的 WHERE
proj = {(t[1],) for t in r}                     # π_[dept](r)          → SQL 的 SELECT dept
print("σ dept='統計' →", sel)
print("π (dept)      →", proj, "← 只剩 2 個元素！")
print("因為 relation 是「集合」，投影後重複自動消失——這就是 SELECT DISTINCT 的數學原意。")

## 0.2 Key：一列資料的身分證

| 名詞 | 定義 | 例（student） |
|---|---|---|
| superkey | 能唯一識別一列的欄位集合 | {sid}、{sid, name}、全欄位 |
| candidate key | **極小**的 superkey（拿掉任一欄就不唯一） | {sid}；若身分證欄存在也是 |
| **primary key** | 從 candidate key 中**選定**的那一個 | sid |
| **foreign key** | 參照另一張表（或自身）符合要求的主鍵／UNIQUE 鍵的欄位組 | takes.sid → student.sid |

小練習：`takes(sid, cid, semester, grade)`（誰、哪門課、哪學期、幾分）的合理主鍵是？

<details><summary>答案</summary>

`{sid, cid, semester}` 複合主鍵——同一人同一課同一學期只能有一筆；只用 `{sid, cid}` 就擋掉重修了。**主鍵的選擇是業務規則的宣告**，不是技術細節。
</details>

In [ ]:
# key 是「語意」的宣告，資料只能「否證」它：用 pandas 檢查候選 key 有沒有被資料打臉
df_s = pd.DataFrame([("S001", "林佳蓉", "統計", 3), ("S002", "陳威廷", "統計", 3),
                     ("S003", "張雅筑", "統計", 3), ("S001", "林佳蓉", "資訊", 2)],   # 故意讓 S001 出現兩次！
                    columns=["sid", "name", "dept", "year"])
for cand in [["sid"], ["name"], ["sid", "dept"]]:
    dups = int(df_s.duplicated(subset=cand).sum())
    print(f"{str(cand):16s} 重複 {dups} 筆 →",
          "❌ 被資料否證，不是 key" if dups else "資料沒打臉（注意：這不構成證明！）")
print("\n→ {sid} 被否證了？要嘛這批資料有鬼（重複學號），要嘛你對業務的理解有鬼。")
print("   key 的最終依據是業務規則；宣告成 PRIMARY KEY 之後，資料庫會替你「永遠」守住它。")

In [ ]:
# 小練習的驗證：takes 的主鍵少一欄會發生什麼事——「重修」被誤判成重複
df_takes = pd.DataFrame([("S001", "C101", "114-1", 58),
                         ("S001", "C101", "115-1", 76),    # 這是重修：合法！
                         ("S001", "C104", "114-1", 92)],
                        columns=["sid", "cid", "semester", "grade"])
print("以 {sid, cid} 當 key      → 重複",
      int(df_takes.duplicated(subset=["sid", "cid"]).sum()), "筆：重修被誤殺（key 太小，擋掉合法資料）")
print("以 {sid, cid, semester} 當 key → 重複",
      int(df_takes.duplicated(subset=["sid", "cid", "semester"]).sum()), "筆：這才是對的主鍵")
print("→ 主鍵選誰＝宣告「什麼情況算同一筆」。選錯的代價是資料進不來、或髒資料進得來。")

In [ ]:
# NULL 初體驗：資料庫的「未知」不是 0、不是空字串——一般相等比較也可能得到 UNKNOWN
mcon = sqlite3.connect(":memory:")
print("NULL = NULL  →", mcon.execute("SELECT NULL = NULL").fetchone()[0], "（None＝UNKNOWN！）")
print("NULL IS NULL →", mcon.execute("SELECT NULL IS NULL").fetchone()[0], "（要用 IS 才問得到）")
print("1 + NULL     →", mcon.execute("SELECT 1 + NULL").fetchone()[0], "（NULL 會傳染）")
print("→ 統計人請把 NULL 想成 missing data。判斷缺值要用 IS NULL；一般相等比較不會得到 TRUE。")

# 第 1 節：定義與維護資料

## 1.1 SQLite 的型別系統：先說清楚它的「怪」

**SQLite 用法**：普通 SQLite 表採彈性型別，值自己帶儲存類別，欄位宣告決定「偏好」（type affinity）。PostgreSQL 的欄位有固定型別，也會做允許的自動轉換，但不會把無法轉成整數的文字原樣存進 INTEGER 欄。

| 儲存類別 | 放什麼 |
|---|---|
| `NULL` | 未知 |
| `INTEGER` | 64-bit 整數 |
| `REAL` | 浮點數 |
| `TEXT` | 文字（SQLite 可使用 UTF-8／UTF-16 編碼） |
| `BLOB` | 原始位元組（圖片、檔案） |

沒有獨立的 DATE／BOOLEAN 型別：日期慣例存 `TEXT`（ISO 格式 `'2026-09-17'`，可比較可排序），布林存 0／1。

- 宣告 `INTEGER` 的欄位塞 `'123'` → 自動轉成整數；塞 `'abc'` → **原樣存成文字，不報錯**。
- 宣告型別與約束有助於表達意圖，但**不能因此保證跨資料庫無痛移植**。SQLite 的 `VARCHAR(10)` 不檢查長度，`NUMERIC` 是 affinity，不是固定精度十進位保證；SQLite 的 INTEGER 值可用 64 位元、REAL 使用雙精度，PG 的 INTEGER 為 32 位元、REAL 為單精度。
- `INTEGER`、`REAL`、`NUMERIC` 是標準型別名稱；`TEXT` 是 SQLite／PG 都有的型別名稱，不能把 SQLite 的五種儲存類別當成 ANSI 型別清單。

依據：[SQLite 型別與比較規則](https://www.sqlite.org/datatype3.html)、[PostgreSQL 數值型別](https://www.postgresql.org/docs/18/datatype-numeric.html)。

In [ ]:
# 動態型別現場：typeof() 看每個值真正的儲存型別
con.execute("DROP TABLE IF EXISTS t_demo")
con.execute("CREATE TABLE t_demo(x INTEGER)")
con.executemany("INSERT INTO t_demo VALUES (?)", [(123,), ("456",), ("abc",), (7.5,), (None,)])
con.commit()
q("SELECT x, typeof(x) FROM t_demo")

### affinity 是怎麼判定的？（AI 生出奇怪型別名時你要看得懂）

SQLite 拿「宣告的型別**字串**」比對關鍵字，由上而下第一個中的算數：

| 規則（依序） | 判為 | 例 |
|---|---|---|
| 含 `INT` | INTEGER | `BIGINT`、`INT8` |
| 含 `CHAR`／`CLOB`／`TEXT` | TEXT | `VARCHAR(10)`、`NCHAR` |
| 含 `BLOB` 或沒宣告 | BLOB（原樣存） | `BLOB` |
| 含 `REAL`／`FLOA`／`DOUB` | REAL | `FLOAT`、`DOUBLE` |
| 其他 | NUMERIC（能轉數字就轉） | `STRING`、`DECIMAL` |

冷知識：宣告 `STRING` 會落到 **NUMERIC**（不含任何關鍵字）——`'123'` 存進去會變整數！可用 `typeof()` 檢查實際儲存的型別。

In [ ]:
# affinity 判定實測：五種宣告、同一個值 '123'，存出五種結果
con.execute("DROP TABLE IF EXISTS aff")
con.execute("CREATE TABLE aff(a VARCHAR(10), b FLOAT, c BIGINT, d BLOB, e STRING)")
con.execute("INSERT INTO aff VALUES ('123','123','123','123','123')")
con.commit()
print(q("SELECT a, typeof(a) AS ta, b, typeof(b) AS tb, c, typeof(c) AS tc, "
        "d, typeof(d) AS td, e, typeof(e) AS te FROM aff").to_string(index=False))
print("\n→ VARCHAR→text、FLOAT→real、BIGINT→integer、BLOB→原樣（text）、STRING→integer（！）")
print("   SQLite 宣告請表達實際用途；INTEGER / REAL / TEXT / BLOB / NUMERIC 在此是 affinity 類別，不是標準 SQL 型別全集。")

### 【選做／加碼】`'10' > '9'`？比較前到底轉了誰

這題不能只背「SQLite 會自動轉型」。要把三件事拆開：

1. **storage class 屬於值**：每個值當下是 NULL、INTEGER、REAL、TEXT 或 BLOB，可用 `typeof()` 看。
2. **affinity 屬於欄位（以及部分運算式）**：寫入欄位或拿欄位比較時，SQLite 會依 affinity **嘗試**轉換；它不是永久、強制的型別宣告。
3. **比較分兩階段**：先依兩邊的 affinity 規則嘗試轉換，再比較。兩邊都是數值就數值比；兩邊都是 TEXT 就依字串順序比；若儲存類別仍不同，排序是 `NULL < INTEGER/REAL < TEXT < BLOB`。

所以，沒有欄位參與時，兩個字串 literal 都沒有 affinity：`'10' > '9'` 是字串比較，答案是 0；甚至 `10 > '9'` 也不會自動把右邊轉成數字，而是 INTEGER 排在 TEXT 前，所以仍是 0。反過來 `'10' > 9` 卻是 1，只因 TEXT 排在數值後面，**不是**因為它把 `'10'` 當成十。若語意確定是數量，請明寫 `CAST`。

欄位加入後才看得到 affinity 的影響：NUMERIC 欄與 `'9'` 比較，會把可轉換的文字套用 NUMERIC affinity；TEXT 欄與數字 `9` 比較，則會把右邊轉成文字。比較結果必須連同操作數的型別一起解讀。

In [ ]:
# 【選做／加碼】literal 沒有欄位 affinity；先猜四個結果再執行
comparison_cases = [
    ("TEXT vs TEXT", "SELECT '10' > '9'"),
    ("INTEGER vs TEXT", "SELECT 10 > '9'"),
    ("TEXT vs INTEGER", "SELECT '10' > 9"),
    ("explicit CAST", "SELECT CAST('10' AS INTEGER) > CAST('9' AS INTEGER)"),
]
for case_name, sql in comparison_cases:
    result = con.execute(sql).fetchone()[0]
    print(f"{case_name:16s} → {result}")

con.execute("DROP TABLE IF EXISTS affinity_compare")
con.execute("CREATE TABLE affinity_compare(num_value NUMERIC, text_value TEXT)")
con.executemany("INSERT INTO affinity_compare VALUES (?,?)", [("10", "10"), ("9", "9")])
con.commit()
print("\n同樣輸入 '10' 與 '9'，寫入後的 storage class 與比較結果：")
q("""SELECT num_value, typeof(num_value) AS num_type,
            num_value > '9' AS num_gt_9,
            text_value, typeof(text_value) AS text_type,
            text_value > 9 AS text_gt_9
     FROM affinity_compare ORDER BY num_value""")
# 重點：不是「SQLite 永遠把數字字串轉成數字」；有沒有欄位 affinity，答案可能不同。

### 隨堂練習 A（先猜再跑）

`CREATE TABLE p(x DOUBLE, y CLOB, z INT8)`，塞進 `('9', '9', '9')`——三個 typeof 各是什麼？

<details><summary>驗證程式與答案</summary>

```python
con.execute("DROP TABLE IF EXISTS p")
con.execute("CREATE TABLE p(x DOUBLE, y CLOB, z INT8)")
con.execute("INSERT INTO p VALUES ('9','9','9')")
print(q("SELECT typeof(x), typeof(y), typeof(z) FROM p"))
```
`real`、`text`、`integer`——DOUBLE 含 DOUB、CLOB 含 CLOB、INT8 含 INT。規則表就是這樣查的。
</details>

In [ ]:
# SQLite 3.37+ 的 STRICT：不能無損轉成宣告型別時拒絕；不是拒絕所有轉型
if sqlite3.sqlite_version_info >= (3, 37):
    con.execute("DROP TABLE IF EXISTS strict_demo")
    con.execute("CREATE TABLE strict_demo(x INTEGER) STRICT")
    con.execute("INSERT INTO strict_demo VALUES ('123')")
    assert con.execute("SELECT x, typeof(x) FROM strict_demo").fetchone() == (123, "integer")
    print("字串 123 可無損轉成 INTEGER，因此接受")
    try:
        con.execute("INSERT INTO strict_demo VALUES ('abc')")
        print("⚠️ 竟然過了？！")
    except sqlite3.IntegrityError as e:
        print("✅ STRICT 表擋下 →", e)
    con.commit()
else:
    print("這顆 SQLite <3.37 沒有 STRICT 表——知道有這功能即可")
# STRICT 是 SQLite 擴充，型別名稱限 INT / INTEGER / REAL / TEXT / BLOB / ANY。
# 普通表的型別宣告不保證型別安全；必填仍需 NOT NULL，值域仍需 CHECK。

## 1.2 `CREATE TABLE`：把「資料的規則」宣告出來

從文件表的基本 CREATE 出發，這一層增加「允許存入什麼資料」的規則；不用把每個約束當成另一種建表命令。

**標準 SQL**：PK、NOT NULL、UNIQUE、CHECK、DEFAULT 與 FK 的基本形式。單欄規則可接在欄位後，多欄規則寫在表層；也可用 `CONSTRAINT constraint_name` 命名。DEFAULT 是提供初值的規則，本身不拒絕不合法的資料。此例的自動編號與 `datetime()` 則是 SQLite 用法。

以下是概念 schema；父表 `member(member_id INTEGER PRIMARY KEY)` 與 `event(event_id INTEGER PRIMARY KEY)` 是社團會員、活動的識別表，須存在後才能寫入合法報名。

```sql
CREATE TABLE registration(                    -- 以社團活動報名為例
  reg_id   INTEGER PRIMARY KEY,               -- 自動編號
  member_id INTEGER NOT NULL REFERENCES member(member_id),
  event_id  INTEGER NOT NULL REFERENCES event(event_id),
  status    TEXT NOT NULL DEFAULT '報名'
            CHECK (status IN ('報名','候補','取消')),
  reg_time  TEXT NOT NULL DEFAULT (datetime('now','+8 hours')),
  UNIQUE (member_id, event_id)                -- 同人同活動只能報一次
);
```

| 約束 | 保證 | 沒有它的慘案 |
|---|---|---|
| `PRIMARY KEY` | 標準語意為唯一＋非空；SQLite 普通表的文字／複合主鍵需補 NOT NULL | 同學號兩筆，成績算誰的？ |
| `NOT NULL` | 必填 | 沒姓名的學生 |
| `UNIQUE` | 不重複（可多欄組合） | 同人重複報名 |
| `CHECK (…)` | 值域／商業規則 | 年級 17、名額 −3 |
| `DEFAULT` | 預設值（可以是運算式） | 每筆都要手填時間 |
| `FOREIGN KEY` | 參照完整性 | 幽靈學生的報名 |

**SQLite 用法**：每條連線都在開始交易前明寫 `PRAGMA foreign_keys = ON;`，並查回值確認為 1，不依賴編譯時的預設。交易中切換不會生效。PostgreSQL 一般外鍵不需要這個開關。

`CHECK` 的條件為 NULL 也會通過；必填需另外寫 NOT NULL。普通 SQLite 表的 `INTEGER CHECK(fee >= 0)` 仍可能接受文字，因為 affinity 不等於強制型別。嚴格型別需求可採 STRICT，或明確檢查儲存類別。[SQLite CREATE TABLE](https://www.sqlite.org/lang_createtable.html)、[STRICT](https://www.sqlite.org/stricttables.html)

In [ ]:
# 約束是「自動守門員」：一條一條踩給你看
con.execute("DROP TABLE IF EXISTS club")
con.execute("""
CREATE TABLE club(
  club_id  INTEGER PRIMARY KEY,
  cname    TEXT NOT NULL UNIQUE,
  captain  TEXT NOT NULL REFERENCES student(sid),
  fee      INTEGER NOT NULL DEFAULT 0 CHECK (fee >= 0)
)""")
con.execute("INSERT INTO club(cname, captain) VALUES ('資料科學社', 'S001')")

violations = [
    ("NOT NULL", "INSERT INTO club(cname, captain) VALUES (NULL, 'S002')"),
    ("UNIQUE",   "INSERT INTO club(cname, captain) VALUES ('資料科學社', 'S003')"),
    ("CHECK",    "INSERT INTO club(cname, captain, fee) VALUES ('登山社', 'S004', -100)"),
    ("FK",       "INSERT INTO club(cname, captain) VALUES ('吉他社', 'S999')"),   # 幽靈學生
]
for label, sql in violations:
    try:
        con.execute(sql)
        print(f"⚠️ {label}：竟然過了？！")
    except sqlite3.IntegrityError as e:
        print(f"✅ {label:8s} 擋下 → {e}")
con.commit()
q("SELECT * FROM club")   # 資料乾乾淨淨，只有合法的那一筆

# 本格驗證列出的四種違規；普通 SQLite 表的數值型別仍有 affinity 的限制。
# PostgreSQL 若在同一交易故意觸發錯誤，需 rollback 或 SAVEPOINT，不能只 catch 後繼續。

> **📌 專題連結**：翻開你的題目找「必備功能」，幾乎每一條都是約束：
> 報名系統的名額 `CHECK(quota >= 0)`、訂票系統的同座位 `UNIQUE(showtime_id, seat_id)`、訂房系統的 `CHECK(check_in < check_out)`、問卷系統的防重複填答 `UNIQUE(survey_id, respondent_id)`⋯⋯
> **能用約束擋的，就不要只靠 Python 的 if**——約束是最後一道防線，多人同時操作時只有它靠得住。

In [ ]:
# DEFAULT 是標準功能；datetime() 及其修飾詞是 SQLite，這種預設運算式需要括號
con.execute("DROP TABLE IF EXISTS notices")
con.execute("""CREATE TABLE notices(
  id      INTEGER PRIMARY KEY,
  body    TEXT NOT NULL,
  created TEXT NOT NULL DEFAULT (datetime('now','+8 hours')))""")
con.execute("INSERT INTO notices(body) VALUES ('週四停課一次')")     # 完全沒提 created
con.execute("INSERT INTO notices(body) VALUES ('教室改 B203')")
con.commit()
q("SELECT * FROM notices")
# 少了括號 DEFAULT datetime('now') 會直接語法錯誤——這是 SQLite 的規定：運算式預設值要括起來

# DEFAULT 只在省略欄位時提供初值，通常不會取代明寫的 NULL。
# DEFAULT CURRENT_TIMESTAMP 是兩家都有的形式，但回傳型別與時間語意仍不同。

In [ ]:
# CHECK 可以跨欄位：把「起 < 訖」這種規則寫進表裡
con.execute("DROP TABLE IF EXISTS room_use")
con.execute("""CREATE TABLE room_use(
  room TEXT NOT NULL,
  s    TEXT NOT NULL,
  e    TEXT NOT NULL,
  CHECK (s < e))""")                     # 跨欄位規則：起訖顛倒直接擋
con.execute("INSERT INTO room_use VALUES ('圓桌室', '2026-09-17 10:00', '2026-09-17 12:00')")
try:
    con.execute("INSERT INTO room_use VALUES ('圓桌室', '2026-09-17 15:00', '2026-09-17 14:00')")
except sqlite3.IntegrityError as e:
    print("✅ 起訖顛倒被擋 →", e)
con.commit()
q("SELECT * FROM room_use")

### 【選做／加碼】CHECK + GLOB：把編碼格式也寫進 schema

**SQLite 用法**：CHECK 是標準 SQL，但 GLOB 是方言。SQLite 核心沒有內建通用的 regular expression；簡單、固定長度的格式可用 GLOB。它是 **glob pattern，不是 regex**：

| 符號 | 意思 |
|---|---|
| <code>*</code> | 任意長度的任意字元 |
| <code>?</code> | 恰好一個任意字元 |
| <code>[0-9]</code> | 恰好一個 ASCII 數字 |

例如 <code>S[0-9]*</code> **不等於**「S 後面全是數字」：它只要求 S 後先有一個數字，後面的星號連英文字也吃，所以 <code>S1xyz</code> 會通過。要驗證「S 加三位數」應完整寫成 <code>S[0-9][0-9][0-9]</code>，沒有星號，長度也一起鎖住。

另一個關鍵：SQLite 的 CHECK 只在結果為 0 時拒絕；結果是 NULL 也算通過。因此必填格式一定要搭配 NOT NULL。GLOB 只能驗證字面形狀；像 <code>2026-99-99</code> 即使符合日期外形，也不是有效日期，語意規則仍要另外處理。

PostgreSQL 可用 `code ~ '^S[0-9]{3}$'` 表達此格式；`~` 也屬該引擎的正規表示式運算子，不應標為 ANSI 的 GLOB 替代語法。

In [ ]:
# 【選做／加碼】先看寬鬆 pattern 的漏洞，再讓 CHECK 擋住格式錯誤
loose_pattern = "S[0-9]*"
exact_pattern = "S[0-9][0-9][0-9]"
samples = ["S001", "S1xyz", "s001", "S01", "S0001", "S12A"]
for sample in samples:
    loose_ok, exact_ok = con.execute(
        "SELECT ? GLOB ?, ? GLOB ?",
        (sample, loose_pattern, sample, exact_pattern),
    ).fetchone()
    print(f"{sample:5s}  loose={loose_ok}  exact={exact_ok}")

con.execute("DROP TABLE IF EXISTS member_code_demo")
con.execute("""CREATE TABLE member_code_demo(
  code TEXT PRIMARY KEY NOT NULL
       CHECK (code GLOB 'S[0-9][0-9][0-9]'))""")
for candidate in ["S001", "S999", "s001", "S01", "S0001", "S12A", "S001x", None]:
    try:
        con.execute("INSERT INTO member_code_demo VALUES (?)", (candidate,))
        print(f"✅ 接受 {candidate!r}")
    except sqlite3.IntegrityError as e:
        print(f"⛔ 擋下 {candidate!r}: {e}")
con.commit()
q("SELECT * FROM member_code_demo ORDER BY code")

In [ ]:
# 複合 UNIQUE：規則是「組合不重複」，單欄可以重複
con.execute("DROP TABLE IF EXISTS seat_pick")
con.execute("""CREATE TABLE seat_pick(
  showtime INTEGER NOT NULL,
  seat     TEXT NOT NULL,
  buyer    TEXT NOT NULL,
  UNIQUE (showtime, seat))""")             # 同場次同座位只能賣一次；不同場次同座位 OK
con.executemany("INSERT INTO seat_pick VALUES (?,?,?)",
                [(1, "A1", "佳蓉"), (2, "A1", "威廷")])       # A1 在兩個場次各賣一次 ✔
try:
    con.execute("INSERT INTO seat_pick VALUES (1, 'A1', '孟軒')")   # 場次 1 的 A1 再賣 → 擋
except sqlite3.IntegrityError as e:
    print("✅ 同場次同座位被擋 →", e)
con.commit()
q("SELECT * FROM seat_pick")
# 「同一 X 的 Y 不可重複」＝ UNIQUE(X, Y)——訂票、預約、報名全是這一句

### FK 的參照動作：父列刪除或鍵值改變時怎麼辦

**標準 SQL**：在 REFERENCES 後指定 ON DELETE 或 ON UPDATE。SQLite 的規則如下：

| 動作 | 行為 |
|---|---|
| `NO ACTION`（預設） | 不自動處理子列；在約束的檢查時點若仍有違規，就拒絕操作 |
| `RESTRICT` | 存在參照子列時立即拒絕父鍵刪改；即使 FK 延後檢查也一樣 |
| `CASCADE` | 刪父列時連鎖刪子列；改父鍵時同步改子鍵 |
| `SET NULL` | 子鍵設為 NULL；相關欄位不能又要求 NOT NULL |
| `SET DEFAULT` | 子鍵設為預設值；預設值仍必須滿足外鍵規則 |

NO ACTION 與 RESTRICT 不是同義詞：`DEFERRABLE INITIALLY DEFERRED` 可將外鍵的最終檢查延到 COMMIT，但 RESTRICT 仍會立即拒絕。延後檢查適合在同一交易內調整互相參照的資料。

外鍵可參照符合要求的 PK 或 UNIQUE 鍵，也可由多欄構成。SQLite 一般 FK 若子鍵含 NULL，就不要求找到父列；關係必填須另外加 NOT NULL。依資料生命週期選動作，不要把 CASCADE 當固定模板。[SQLite 外鍵規則](https://www.sqlite.org/foreignkeys.html)

In [ ]:
# CASCADE 現場：刪一篇貼文，它的留言「連坐」蒸發
con.executescript("""
DROP TABLE IF EXISTS comments; DROP TABLE IF EXISTS posts;
CREATE TABLE posts(pid INTEGER PRIMARY KEY, title TEXT NOT NULL);
CREATE TABLE comments(
  cid INTEGER PRIMARY KEY,
  pid INTEGER NOT NULL REFERENCES posts(pid) ON DELETE CASCADE,
  txt TEXT NOT NULL);
INSERT INTO posts(pid, title) VALUES (1, '資料庫好好玩'), (2, '求救：NULL 是什麼');
INSERT INTO comments(pid, txt) VALUES (1, '推'), (1, '先收藏'), (2, '用 IS NULL 啦');
""")
print("刪除前：", q("SELECT COUNT(*) c FROM comments").iloc[0, 0], "則留言")
con.execute("DELETE FROM posts WHERE pid = 1")
con.commit()
print("刪掉貼文 1 之後：")
print(q("SELECT * FROM comments").to_string(index=False))
print("→ 貼文 1 的兩則留言連鎖消失。方便，但也請想像誤刪整個社團時的畫面——所以預設值是「擋下」。")

## 1.3 主鍵的三種常見長相

| 寫法 | 行為 | 用在哪 |
|---|---|---|
| `sid TEXT PRIMARY KEY NOT NULL` | 自然鍵：業務本來就有的唯一編號 | 學號、ISBN、身分證 |
| `id INTEGER PRIMARY KEY` | **代理鍵**：SQLite 幫你自動編號（就是內部 rowid 的別名） | 訂單、報名、貼文⋯⋯大多數表 |
| `PRIMARY KEY (a, b)` | 複合主鍵 | 多對多關聯表（takes、選課） |

**SQLite 用法**：普通 rowid 表中，型別名稱恰為 `INTEGER` 的單欄 PRIMARY KEY 可作為 rowid 別名並自動編號；`INT PRIMARY KEY` 不等同這個寫法。文字與複合主鍵的欄位則明寫 NOT NULL，避免普通 SQLite 表容許主鍵 NULL 的歷史例外。

`AUTOINCREMENT` 改變自動分配 ID 的規則，避免重用先前已提交列的 ID，但不保證連號；回滾掉的 ID 仍可再用，顯式提供 ID 也不是同一項保證。是否需要由業務決定。[SQLite AUTOINCREMENT](https://www.sqlite.org/autoinc.html)

PostgreSQL 的 `INTEGER PRIMARY KEY` 不會自動編號。標準 identity 語法可寫 `id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY`：identity 產生值，PK 負責唯一／非空。[PostgreSQL identity](https://www.postgresql.org/docs/18/ddl-identity-columns.html)

In [ ]:
# 主鍵非空的反例與防護：獨立連線，不改課程資料
pk_demo = sqlite3.connect(":memory:")
pk_demo.execute("CREATE TABLE loose_key(code TEXT PRIMARY KEY)")  # 故意保留 SQLite 歷史例外
pk_demo.execute("INSERT INTO loose_key VALUES (NULL), (NULL)")
assert pk_demo.execute("SELECT COUNT(*) FROM loose_key").fetchone()[0] == 2
print("普通 TEXT PRIMARY KEY 竟可存兩筆 NULL：", pk_demo.execute("SELECT * FROM loose_key").fetchall())
pk_demo.execute("CREATE TABLE required_key(code TEXT PRIMARY KEY NOT NULL)")
try:
    pk_demo.execute("INSERT INTO required_key VALUES (NULL)")
except sqlite3.IntegrityError:
    print("明寫 NOT NULL 後，缺少主鍵會被拒絕")
else:
    raise AssertionError("主鍵非空規則沒有生效")
pk_demo.close()

In [ ]:
# 代理鍵示範：不給 id，資料庫自己編
con.execute("DROP TABLE IF EXISTS post")
con.execute("CREATE TABLE post(post_id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
for t in ["第一篇", "第二篇", "第三篇"]:
    cur = con.execute("INSERT INTO post(title) VALUES (?)", (t,))
    print(f"插入「{t}」 → 自動編號 post_id = {cur.lastrowid}")
con.commit()
q("SELECT * FROM post")

In [ ]:
# 幕後真相：每張（一般）表都有隱藏的 rowid；INTEGER PRIMARY KEY 就是它的別名
print(q("SELECT rowid, post_id, title FROM post").to_string(index=False))
print("→ rowid 與 post_id 同一個東西（別名）。一般 SQLite rowid 表以 rowid 作為資料列的內部識別鍵。")
print("   刪掉最大列再插入，編號可能重用；AUTOINCREMENT 可避免自動重用已提交 ID，但仍不保證連號。")

## 1.4 `INSERT`：三種資料來源與衝突處理

基本 INSERT 不變，只增加「資料從哪裡來」與「遇到重複怎麼辦」兩個問題。多列 VALUES 省下重複命令；UPSERT 則增加衝突時的行為，不只是排版 sugar。

**標準 SQL**：VALUES、SELECT 與 DEFAULT VALUES 是三種基本資料來源。指定欄位清單時，其餘一般欄位使用宣告的 default，沒有 default 才是 NULL；自動編號與 generated 欄位另依其產生值規則。多列 VALUES 是一句 SQL；Python 的 `executemany()` 則是用多組參數執行同一模板。

```sql
INSERT INTO club(cname, captain) VALUES ('桌遊社','S002'), ('熱舞社','S003');  -- 一次多列
INSERT INTO takes(sid, cid, semester)                                          -- 從查詢結果塞入
  SELECT sid, 'C103', '115-1' FROM student WHERE dept = '統計' AND year = 4;
INSERT INTO club DEFAULT VALUES;                                               -- 語法示例：此表缺 cname/captain，會違反 NOT NULL
```

**共有擴充**：`ON CONFLICT ... DO NOTHING/DO UPDATE` 不是 ANSI SQL；SQLite 3.24+ 沿用 PostgreSQL 的 UPSERT 語法。它針對唯一性衝突選擇替代動作，不是一般資料驗證器：候選列若被 DO NOTHING 略過，不能因此推定它滿足所有其他規則；例如未實際插入的列可能不觸發 FK 檢查。[SQLite UPSERT](https://www.sqlite.org/lang_upsert.html)

In [ ]:
# 省略欄位、DEFAULT VALUES、明寫 NULL 是不同操作
default_demo = sqlite3.connect(":memory:")
default_demo.execute("CREATE TABLE defaults_demo(id INTEGER PRIMARY KEY, qty INTEGER DEFAULT 7)")
default_demo.execute("INSERT INTO defaults_demo DEFAULT VALUES")
default_demo.execute("INSERT INTO defaults_demo(qty) VALUES (NULL)")
default_rows = default_demo.execute("SELECT qty FROM defaults_demo ORDER BY id").fetchall()
print("省略 qty / 明寫 NULL：", default_rows)
assert default_rows == [(7,), (None,)]
default_demo.execute("CREATE TABLE required_demo(name TEXT NOT NULL)")
try:
    default_demo.execute("INSERT INTO required_demo DEFAULT VALUES")
except sqlite3.IntegrityError:
    print("DEFAULT VALUES 仍須通過 NOT NULL；沒有可用的預設值就失敗")
else:
    raise AssertionError("必填欄位沒有被檢查")
default_demo.close()

In [ ]:
# UPSERT 之一（DO NOTHING）：打卡系統——同一人同一天重複打卡，不報錯、也不重複
con.execute("DROP TABLE IF EXISTS checkin")
con.execute("""CREATE TABLE checkin(
  sid TEXT NOT NULL REFERENCES student(sid),
  d   TEXT NOT NULL,                        -- 日期
  t   TEXT,                        -- 第一次成功寫入的打卡時間
  PRIMARY KEY (sid, d))""")

def punch_in(sid, d, t):
    con.execute("""INSERT INTO checkin VALUES (?,?,?)
                   ON CONFLICT(sid, d) DO NOTHING""", (sid, d, t))

punch_in("S001", "2026-09-17", "08:55")
punch_in("S001", "2026-09-17", "13:10")   # 同一天第二次 → 靜默忽略，保留第一次成功寫入那筆，不比較 t 的早晚
punch_in("S002", "2026-09-17", "09:02")
con.commit()
q("SELECT * FROM checkin")

In [ ]:
# UPSERT 之二（DO UPDATE）：頁面點閱計數——第一次插入、之後累加，一句搞定
con.execute("DROP TABLE IF EXISTS page_views")
con.execute("CREATE TABLE page_views(page TEXT PRIMARY KEY NOT NULL, hits INTEGER NOT NULL DEFAULT 1)")

def visit(page):
    con.execute("""INSERT INTO page_views(page) VALUES (?)
                   ON CONFLICT(page) DO UPDATE SET hits = page_views.hits + 1""", (page,))

for p in ["首頁", "首頁", "課表", "首頁", "成績", "課表"]:
    visit(p)
con.commit()
q("SELECT * FROM page_views ORDER BY hits DESC")
# page_views.hits 是既有值；表名前綴也避免 PostgreSQL 的模糊參照。
# excluded.hits 是這次原本要插入的值；SET hits = excluded.hits 表示覆寫，不是累加。

### `OR IGNORE` 不等於具名目標的 UPSERT `DO NOTHING`

SQLite 有兩套長得像、但**作用範圍不同**的寫法：

| 寫法 | 衝突目標 | 遇到其他壞資料 |
|---|---|---|
| `INSERT OR IGNORE ...` | 沒有 target；是整句 INSERT 的 SQLite conflict policy | UNIQUE／PK／NOT NULL／CHECK 等可適用的違規都可能被**靜默略過**；FK 違規仍會報錯 |
| `... ON CONFLICT(sid, d) DO NOTHING` | 針對指定的 UNIQUE／PK 衝突選擇略過 | 不是通用忽略策略，但被略過的候選列也不代表通過全部資料驗證 |

所以前面打卡例子用 UPSERT：它說清楚「只忽略同人同日」，不等於把整句改成 OR IGNORE。不過撞到目標鍵而被略過的列，可能根本不進入 FK 檢查；不能拿 DO NOTHING 當作輸入驗證。

`INSERT OR REPLACE` 更不是 update：撞到 UNIQUE／PK 時會**刪掉舊列，再插入新列**。因此 rowid 可能變、沒給的欄位回到 DEFAULT，連鎖 FK 還可能把子列一起刪掉。AI 寫出 `OR REPLACE` 時，別把它當成無害的 UPSERT。

In [ ]:
# 親眼比較：OR IGNORE 可吞掉 NOT NULL；指定 target 的 DO NOTHING 不會
con.executescript("""
DROP TABLE IF EXISTS policy_demo;
CREATE TABLE policy_demo(k TEXT PRIMARY KEY NOT NULL, required_text TEXT NOT NULL);
INSERT INTO policy_demo VALUES ('A', 'ok');
""")
con.execute("INSERT OR IGNORE INTO policy_demo VALUES ('B', NULL)")
print("OR IGNORE 後有 B 嗎？", con.execute("SELECT COUNT(*) FROM policy_demo WHERE k='B'").fetchone()[0])
try:
    con.execute("""INSERT INTO policy_demo VALUES ('B', NULL)
                   ON CONFLICT(k) DO NOTHING""")
except sqlite3.IntegrityError as e:
    print("指定 k 的 DO NOTHING 不會吞 NOT NULL 錯誤 →", e)

# OR REPLACE 的「刪＋插」：舊 id 消失、DEFAULT 重設、ON DELETE CASCADE 刪掉子列
con.executescript("""
DROP TABLE IF EXISTS replace_child; DROP TABLE IF EXISTS replace_parent;
CREATE TABLE replace_parent(
  id INTEGER PRIMARY KEY, code TEXT UNIQUE, label TEXT NOT NULL DEFAULT '預設');
CREATE TABLE replace_child(
  id INTEGER PRIMARY KEY, parent_id INTEGER REFERENCES replace_parent(id) ON DELETE CASCADE);
INSERT INTO replace_parent(code, label) VALUES ('A', '舊說明');
INSERT INTO replace_child(parent_id) VALUES (last_insert_rowid());
""")
old_id = con.execute("SELECT id FROM replace_parent WHERE code='A'").fetchone()[0]
con.execute("INSERT OR REPLACE INTO replace_parent(code) VALUES ('A')")
new_row = con.execute("SELECT id, code, label FROM replace_parent WHERE code='A'").fetchone()
child_count = con.execute("SELECT COUNT(*) FROM replace_child").fetchone()[0]
print("OR REPLACE：舊 id =", old_id, "→ 新列 =", new_row, "；剩下子列 =", child_count)
assert new_row[0] != old_id and new_row[2] == '預設' and child_count == 0
con.commit()
print("→ 真正要保留同一列及其子列，請用 ON CONFLICT(...) DO UPDATE。")

In [ ]:
# INSERT ... SELECT：把查詢結果整批搬家（畢業生歸檔）
con.execute("DROP TABLE IF EXISTS alumni")
con.execute("CREATE TABLE alumni(sid TEXT PRIMARY KEY NOT NULL, name TEXT, dept TEXT)")
n = con.execute("INSERT INTO alumni SELECT sid, name, dept FROM student WHERE year = 4").rowcount
con.commit()
print(f"歸檔 {n} 位（期望 5）")
q("SELECT * FROM alumni ORDER BY sid")

In [ ]:
# 共有擴充 RETURNING：SQLite 3.35+ 與 PostgreSQL 都有；不是 ANSI 標準
if sqlite3.sqlite_version_info >= (3, 35):
    row = con.execute("INSERT INTO post(title) VALUES ('第四篇') RETURNING post_id, title").fetchone()
    con.commit()
    print("剛插入的列：", row)
else:
    print("這顆 SQLite <3.35 沒有 RETURNING；用 cur.lastrowid 拿自動編號即可")
# 應用場景：新增訂單後立刻要訂單編號去建明細——RETURNING 一趟搞定

# INSERT / UPDATE / DELETE 都可搭配 RETURNING；它回傳本句直接處理的列。
# SQLite 不能像 PostgreSQL 把 INSERT ... RETURNING 放進 CTE 當另一個查詢的資料來源。

## 1.5 `UPDATE`／`DELETE`：鐵律——先想 WHERE

文件表例子是 SET 一個固定檔名；這一層把右側換成計算式或一次指定多欄，WHERE 挑列的責任不變。

```sql
UPDATE takes SET grade = 61 WHERE sid='S006' AND cid='C101' AND semester='114-1';
DELETE FROM club WHERE cname = '熱舞社';
```

忘記 `WHERE` 會修改或刪除整張表。用 `BEGIN` 開始交易、檢查結果，未提交時可用 `ROLLBACK` 撤回修改。

**標準 SQL**：SET 可以一次指定多欄，右側可以是常數、原值運算或 CASE。SQLite／PG 的各個 SET 右側都先讀取修改前的值，因此 `SET a=b, b=a` 可交換兩欄。

`rowcount` 是 Python driver 回報本句處理的列數；即使 `SET x=x` 值未變，也可能算一列。它不等於所有 trigger／FK 級聯變更的總列數。`BEGIN` 是 SQLite／PG 都有的交易起始寫法，標準形式是 `START TRANSACTION`。

In [ ]:
# 「忘記 WHERE」災難現場（在交易裡演，演完回滾，資料毫髮無傷）
con.execute("BEGIN")
n_hit = con.execute("UPDATE takes SET grade = 100").rowcount     # 少了 WHERE！
print(f"UPDATE takes SET grade = 100  →  改掉了 {n_hit} 列（全班全科 100 分）")
print("平均分數變成：", con.execute("SELECT AVG(grade) FROM takes").fetchone()[0])
con.execute("ROLLBACK")                                          # 交易回滾＝時光倒流
print("ROLLBACK 之後平均：", round(con.execute("SELECT AVG(grade) FROM takes").fetchone()[0], 2))
print("→ 實務守則：改資料前先 BEGIN；先用同條件 SELECT 看會動到哪些列。")

In [ ]:
# UPDATE 的值可以是運算式；rowcount 告訴你本句處理幾列，不保證每列值都不同——改完必看的數字
con.execute("BEGIN")
n = con.execute("""UPDATE instructor SET salary = ROUND(salary * 1.05)
                   WHERE dept = '統計' AND salary IS NOT NULL""").rowcount
print(f"統計系調薪 5% → 影響 {n} 列（預期 2：徐教授 salary 是 NULL，不動）")
print(q("SELECT name, salary FROM instructor ORDER BY iid").to_string(index=False))
con.execute("ROLLBACK")     # 課堂示範完回滾；正式要生效改成 con.commit()
print("（已回滾）→ 檢查 rowcount 是應用系統的好習慣：預期改 1 列卻改了 0 或 50 列，就是 bug 的味道。")

In [ ]:
# UPDATE 的多種寫法都沿用「挑列＋算新值」；資料隔離在記憶體
update_demo = sqlite3.connect(":memory:")
update_demo.execute("CREATE TABLE update_values(id INTEGER PRIMARY KEY, qty INTEGER, label TEXT)")
update_demo.executemany("INSERT INTO update_values VALUES (?,?,?)", [(1, 2, "old"), (2, 8, "old")])
update_demo.execute("""UPDATE update_values
    SET qty = qty + 1,
        label = CASE WHEN qty < 5 THEN 'low' ELSE 'ready' END
    WHERE id IN (1, 2)""")
updated = update_demo.execute("SELECT qty, label FROM update_values ORDER BY id").fetchall()
assert updated == [(3, "low"), (9, "ready")]
print("多欄更新：", updated)
unchanged = update_demo.execute("UPDATE update_values SET qty=qty WHERE id=1").rowcount
assert unchanged == 1
print("值沒變也算處理一列：", unchanged)
if sqlite3.sqlite_version_info >= (3, 35):
    returned = update_demo.execute("UPDATE update_values SET qty=qty+1 WHERE id=1 RETURNING id,qty").fetchall()
    assert returned == [(1, 4)]
    print("UPDATE RETURNING：", returned)
update_demo.close()

In [ ]:
# FK 也擋刪除：有修課紀錄的學生不能直接消失
try:
    con.execute("DELETE FROM student WHERE sid = 'S001'")
except sqlite3.IntegrityError as e:
    print(f"✅ 擋下 → {e}")
    print("   設計選項：預設 NO ACTION 檢查不通過時擋下/ ON DELETE CASCADE / SET NULL——1.2 的策略表")
con.commit()

In [ ]:
# 安全刪除三步舞：SELECT 看到 → DELETE 同條件 → rowcount 對帳（跟預覽筆數一樣才安心）
con.execute("INSERT OR IGNORE INTO club(cname, captain) VALUES ('快閃社', 'S002')")
con.commit()
preview = q("SELECT * FROM club WHERE cname = '快閃社'")
print("① 預覽會刪到誰："); print(preview.to_string(index=False))
n = con.execute("DELETE FROM club WHERE cname = '快閃社'").rowcount
con.commit()
print(f"② 執行刪除 → rowcount = {n}")
print("③ 對帳：預覽", len(preview), "列 vs 實刪", n, "列 →", "✅ 一致" if n == len(preview) else "❌ 不對勁！")

### DDL 決策樹（設計時腦中跑一遍）

```
這個欄位…
├─ 一定要有值？ ────────────── NOT NULL
├─ 不能跟別列重複？ ─────────── UNIQUE（組合不重複 → UNIQUE(a, b)）
├─ 有值域／商業規則？ ───────── CHECK（跨欄位也行）
├─ 常常是同一個預設？ ───────── DEFAULT（SQLite 的函數運算式需括號）
├─ 指向別張表的一列？ ───────── REFERENCES ＋ 想好刪除策略
└─ 可以從其他欄算出來？ ─────── generated column（或乾脆別存）
```

## 1.6 generated column：會自己算的欄位

「訂單明細要不要存 `amount = qty × price`？」存了怕不同步、不存每次都要算——
第三條路：宣告成 **generated column**，資料庫保證它永遠等於公式：

```sql
amount REAL GENERATED ALWAYS AS (qty * unit_price)          -- VIRTUAL：查詢時即算（預設）
amount REAL GENERATED ALWAYS AS (qty * unit_price) STORED   -- STORED：寫入時算好存起來
```

（注意：這跟「成交當下的歷史單價」是兩回事——商品現價可能改變，成交時的價格必須獨立保存。）

**標準功能／實作差異**：generated column 的概念與基本子句是標準；STORED／VIRTUAL 是實作選項。SQLite 3.31+ 支援兩者，省略時預設 VIRTUAL；PG 18 支援兩者，PG 12–17 只支援 STORED 且需明寫。公式限同列可決定的運算，不能用子查詢讀商品現價。

DEFAULT 在插入時提供初值；generated column 則維持公式，不應直接塞值。[SQLite generated columns](https://www.sqlite.org/gencol.html)、[PostgreSQL generated columns](https://www.postgresql.org/docs/18/ddl-generated-columns.html)

In [ ]:
# generated column 實測：算好給你，而且不准你亂塞
con.execute("DROP TABLE IF EXISTS line_items")
con.execute("""CREATE TABLE line_items(
  item_id    INTEGER PRIMARY KEY,
  qty        INTEGER NOT NULL CHECK (qty > 0),
  unit_price REAL    NOT NULL CHECK (unit_price >= 0),
  amount     REAL    GENERATED ALWAYS AS (qty * unit_price))""")
con.executemany("INSERT INTO line_items(qty, unit_price) VALUES (?,?)", [(2, 45), (1, 120), (5, 30)])
con.commit()
print(q("SELECT * FROM line_items").to_string(index=False))
try:
    con.execute("INSERT INTO line_items(qty, unit_price, amount) VALUES (1, 10, 999)")   # 想造假帳？
except sqlite3.OperationalError as e:
    print("\n✅ 亂塞被擋 →", e)
print("→ 「可推導的值」交給公式管，永遠不會不同步。")

## 1.7 改表、砍表、看目錄

```sql
ALTER TABLE student ADD COLUMN email TEXT;      -- 加欄位
ALTER TABLE student RENAME COLUMN email TO mail;-- 改欄名（3.25+）
ALTER TABLE student DROP COLUMN mail;           -- 砍欄位（3.35+）
ALTER TABLE student RENAME TO student_old;      -- 改表名
DROP TABLE IF EXISTS t_demo;                    -- 刪除表與資料；已提交後不能再用 ROLLBACK 復原
```

這些操作在 PostgreSQL 也有，但 RENAME、IF EXISTS 等附加形式不要一律稱為標準。SQLite 的 ALTER 能力依版本而異：3.53.0 新增部分 NOT NULL／CHECK 約束的增刪；較舊執行環境不支援。先看 `sqlite3.sqlite_version`，不要把最新版文件當成目前 runtime 的保證。[SQLite 3.53.0 更新說明](https://www.sqlite.org/releaselog/3_53_0.html)

改型別或其他不支援的綱要變更，需依官方流程建新表、搬資料、重建相關索引／trigger／view 並檢查外鍵，不能只換表名。[SQLite ALTER TABLE](https://www.sqlite.org/lang_altertable.html)

DROP TABLE 在明確尚未提交的交易內可 ROLLBACK；一旦已提交，則須靠備份等方式復原。

**SQLite 系統目錄**：schema 本身也存在 `sqlite_schema`（相容名稱 `sqlite_master`）。PG 使用 `information_schema` 或 `pg_catalog`；PRAGMA 不是標準 SQL。

In [ ]:
# ALTER 三招連跳
con.execute("ALTER TABLE post ADD COLUMN note TEXT")
print("加欄後：", [r[1] for r in con.execute("PRAGMA table_info(post)")])
con.execute("ALTER TABLE post RENAME COLUMN note TO memo")
print("改名後：", [r[1] for r in con.execute("PRAGMA table_info(post)")])
if sqlite3.sqlite_version_info >= (3, 35):
    con.execute("ALTER TABLE post DROP COLUMN memo")
    print("砍欄後：", [r[1] for r in con.execute("PRAGMA table_info(post)")])
con.commit()

In [ ]:
# 未提交 DDL 可以回滾；此例只操作獨立記憶體資料庫
ddl_demo = sqlite3.connect(":memory:")
ddl_demo.execute("CREATE TABLE rollback_demo(id INTEGER PRIMARY KEY)")
ddl_demo.execute("INSERT INTO rollback_demo VALUES (1)")
ddl_demo.commit()
ddl_demo.execute("BEGIN")
ddl_demo.execute("DROP TABLE rollback_demo")
ddl_demo.rollback()
assert ddl_demo.execute("SELECT id FROM rollback_demo").fetchall() == [(1,)]
print("ROLLBACK 後表與資料都恢復；前提是 DROP 尚未提交")
ddl_demo.close()

In [ ]:
# 系統目錄：資料庫「知道自己長什麼樣」
print(q("SELECT name, type FROM sqlite_master WHERE type='table' ORDER BY name").to_string(index=False))
print()
print("PRAGMA table_info(student) →")
print(q('SELECT cid, name, type, "notnull", pk FROM pragma_table_info(\'student\')').to_string(index=False))

## 1.8 讀懂錯誤訊息：sqlite3 的例外家族

錯誤訊息不是懲罰，是**資料庫在跟你講話**。Python sqlite3 把錯誤分成幾類，先認臉：

| 例外 | 在說什麼 | 常見案例 |
|---|---|---|
| `OperationalError` | **SQL 本身**有問題 | 表／欄不存在、語法打錯、資料庫被鎖 |
| `IntegrityError` | SQL 沒錯，**資料違規** | NOT NULL／UNIQUE／CHECK／FK 被觸發 |
| `ProgrammingError` | 你跟 **API** 的溝通出錯 | `?` 數量對不上、連線已關閉 |

三步讀法：**① 哪一類**（決定往哪找）→ **② 哪個物件**（訊息裡有表名／欄名／約束名）→ **③ 哪條規則**。
問 AI 除錯時，**把整段 traceback 原文貼上**（別只貼「它報錯了」）——訊息裡的物件名就是答案的一半。

這些是 **Python sqlite3／DB-API 介面**，不是 SQL 語法。`?` 的綁定方式、`execute()`／`executemany()`、`lastrowid` 與交易錯誤後的狀態，都要按 driver 檢查。參數綁定用來提供資料值，不能把表名或欄名當作 `?` 參數。

In [ ]:
# 錯誤動物園：五種常見錯誤各養一隻，看清楚長相
error_zoo = [
    ("查不存在的表", "SELECT * FROM no_such_table"),
    ("查不存在的欄", "SELECT nickname FROM student"),
    ("SQL 打錯字",   "SELEC * FROM student"),
    ("NOT NULL 違規","INSERT INTO club(cname, captain) VALUES (NULL, 'S001')"),
    ("UNIQUE 違規",  "INSERT INTO club(cname, captain) VALUES ('資料科學社', 'S002')"),
]
for label, sql in error_zoo:
    try:
        con.execute(sql)
        print(f"{label}：過了")
    except sqlite3.Error as e:
        print(f"{label:10s} {type(e).__name__:17s}| {e}")
con.rollback()

In [ ]:
# ProgrammingError 最經典的一隻：? 的數量對不上——九成是「忘了逗號的 tuple」
try:
    con.execute("SELECT * FROM student WHERE sid = ?", ("S001"))    # ("S001") 是字串，不是 tuple！
except sqlite3.ProgrammingError as e:
    print("ProgrammingError →", e)
print()
print("修法：('S001',) ——那個逗號就是 tuple 的靈魂：")
print(con.execute("SELECT sid, name FROM student WHERE sid = ?", ("S001",)).fetchone())
# 訊息裡的「there are 4 supplied」＝字串 'S001' 被拆成 4 個字元。看懂訊息，bug 自己招供。

### 隨堂練習 B（寫在紙上或練習工作區）

為**手搖飲料店**的「訂單明細」設計 DDL：
一筆明細屬於某張訂單（`order_id`）、某個品項（`product_id`），有數量（1–20 杯）、單價（≥0）、甜度（限「正常／半糖／微糖／無糖」）。
要求：主鍵、兩個 FK、`CHECK` × 3、合理的 `DEFAULT`。寫完跟隔壁互相挑毛病：**「我可以塞什麼垃圾進你的表？」**

<details><summary>參考解答</summary>

```sql
CREATE TABLE order_item(
  item_id    INTEGER PRIMARY KEY,
  order_id   INTEGER NOT NULL REFERENCES orders(order_id),
  product_id INTEGER NOT NULL REFERENCES product(product_id),
  qty        INTEGER NOT NULL DEFAULT 1 CHECK (qty BETWEEN 1 AND 20),
  unit_price REAL    NOT NULL CHECK (unit_price >= 0),
  sweetness  TEXT    NOT NULL DEFAULT '正常'
             CHECK (sweetness IN ('正常','半糖','微糖','無糖'))
);
```
（也可以加 `UNIQUE(order_id, product_id, sweetness)` 防同單重複列——這是設計判斷，說得出理由就好。
加碼思考：小計欄要用 generated column 嗎？`qty * unit_price` 可以；但 `unit_price` 本身要「存成交當下的價」，別用 FK 即時去撈商品表，否則商品改價就會連帶改變歷史訂單金額。）
</details>

In [ ]:
# 隨堂練習 B 工作區
ex_con = sqlite3.connect(":memory:")

# TODO：CREATE TABLE order_item(...)，然後塞一筆合法、試三筆違規




# 第 2 節：單表查詢的常用組合

## 2.1 `SELECT` 骨架：書寫順序 ≠ 執行順序

```sql
SELECT DISTINCT column_or_expr         -- (5) 挑欄位（投影）
FROM   table_name                            -- (1) 從哪張表
WHERE  row_condition                      -- (2) 逐列過濾
GROUP BY ...  HAVING ...             -- (3)(4) 分組與組過濾
ORDER BY sort_key                      -- (6) 排序
LIMIT  n OFFSET m;                   -- (7) 切前幾筆
```

**標準 SQL**：SELECT、WHERE、DISTINCT、GROUP BY、HAVING、ORDER BY 的基本形式；**共有擴充**：LIMIT／OFFSET。括號裡是**邏輯處理順序**，不是引擎實際必須逐步執行的計畫。`WHERE` 先過濾資料列，`SELECT` 才計算輸出欄位，因此條件與輸出別名的使用位置要分清楚。

In [ ]:
q("SELECT name, dept, year FROM student WHERE dept = '統計' AND year >= 3")

In [ ]:
# 標準寫法：WHERE 重寫運算式；ORDER BY 可使用輸出別名
r = q("SELECT sid, grade, grade * 1.05 AS adj FROM takes WHERE grade * 1.05 > 90 ORDER BY adj DESC")
print(r.head(3).to_string(index=False))
print(f"共 {len(r)} 列（期望 16）")

# SQLite 擴充示範：WHERE 也容許這個不與原欄名重複的別名；PG 不接受
sqlite_alias = q("SELECT sid, grade, grade * 1.05 AS adj FROM takes WHERE adj > 90 ORDER BY adj DESC")
assert r.equals(sqlite_alias)
print("兩種在 SQLite 的結果相同；跨引擎使用上面的標準形式")

In [ ]:
# AND 比 OR 黏——不加括號，條件的意思整個歪掉
no_paren   = q("SELECT * FROM takes WHERE cid='C101' OR cid='C104' AND grade >= 90")
with_paren = q("SELECT * FROM takes WHERE (cid='C101' OR cid='C104') AND grade >= 90")
print(f"WHERE cid='C101' OR cid='C104' AND grade>=90 → {len(no_paren)} 列（期望 15）")
print(f"WHERE (cid='C101' OR cid='C104') AND grade>=90 → {len(with_paren)} 列（期望 4）")
print("→ 上句實際是：C101 全部 ∪（C104 且 ≥90）。OR 出沒，必加括號——AI 生的長 WHERE 尤其要檢查。")

In [ ]:
# BETWEEN（含兩端）／ IN ／ 條件組合
q("""SELECT sid, cid, grade FROM takes
     WHERE grade BETWEEN 85 AND 95
       AND cid IN ('C101','C102','C104')
     ORDER BY grade DESC""")

In [ ]:
# LIKE/ESCAPE 是標準形式；SQLite 預設 ASCII 不分大小寫，GLOB 是分大小寫的 SQLite 用法
print(q("SELECT sid, name FROM student WHERE name LIKE '林%'").to_string(index=False))   # 姓林
print()
print(q("SELECT cid, title FROM course WHERE title LIKE '%統計%'").to_string(index=False))  # 課名含統計
print()
print(q("SELECT sid FROM student WHERE sid GLOB 'S0[01]*'").head(6).to_string(index=False)) # GLOB 可用字元集

In [ ]:
# LIKE 的兩個細節：大小寫、以及「找字面上的 %」
print("'s001' LIKE 'S%'  →", con.execute("SELECT 's001' LIKE 'S%'").fetchone()[0], "（ASCII 不分大小寫！學號比對請先統一大小寫或用 GLOB）")
print("'s001' GLOB 'S*'  →", con.execute("SELECT 's001' GLOB 'S*'").fetchone()[0], "（GLOB 分大小寫）")
print("找「50%off」這種含 % 的字面值 → 用 ESCAPE：")
print("'50%off' LIKE '50\\%%' ESCAPE '\\' →",
      con.execute(r"SELECT '50%off' LIKE '50\%%' ESCAPE '\'").fetchone()[0])

# PostgreSQL 的 LIKE 大小寫行為取決於 collation；常見的確定性設定區分大小寫。
# PostgreSQL 的 ILIKE 是其擴充，不是 SQLite 可直接使用的替代關鍵字。

## 2.2【主線】NULL 專場：三值邏輯（統計系最常摔的坑）

`grade` 是 NULL 表示「在修中」。直覺會這樣寫：

```sql
SELECT * FROM takes WHERE grade = NULL;     -- 回傳 0 列！
```

因為**一般的等號／大小比較只要有 NULL，結果就是 UNKNOWN**（不是 TRUE 也不是 FALSE），而 `WHERE` 只保留 TRUE。

| 規則 | 後果 |
|---|---|
| `x = NULL` → UNKNOWN | 要用 `x IS NULL`／`x IS NOT NULL` |
| NULL 會傳染：`NULL + 1` → NULL | 一般算術運算遇到 NULL，結果也為 NULL；不代表所有函數與邏輯運算都如此 |
| `NOT UNKNOWN` → UNKNOWN | `WHERE NOT (grade > 60)` 一樣撈不到 NULL 列 |
| SUM／AVG／MIN／MAX 等數值聚合**忽略** NULL | `AVG(grade)` 只平均有成績的列 |
| `COUNT(*)` 數列、`COUNT(grade)` 數非 NULL | 兩者的差 ＝ NULL 列數 |
| UNIQUE 欄允許**多個 NULL** 共存 | 「未填 email」不會互撞（SQLite／PostgreSQL 行為） |

> `AVG(x)` 忽略的是 x 的 NULL。`AVG(x), AVG(y)` 可能使用不同列，並不是排除任一欄缺值的整列（listwise deletion）。報表可另列各欄非 NULL 筆數，說清楚各自分母。

UNIQUE 容許多個 NULL 是 SQLite／PG 的預設約束規則，不能解釋成 `NULL = NULL` 為 FALSE；它實際是 UNKNOWN。PG 可另外指定 `NULLS NOT DISTINCT` 改變唯一性判定。

In [ ]:
print("grade = NULL  →", len(q("SELECT * FROM takes WHERE grade = NULL")), "列（陷阱！）")
print("grade IS NULL →", len(q("SELECT * FROM takes WHERE grade IS NULL")), "列（在修中）")
print("NOT (grade >= 60) →", len(q("SELECT * FROM takes WHERE NOT (grade >= 60)")), "列（NULL 列一樣被排除）")
q("""SELECT COUNT(*)                AS total_rows,
            COUNT(grade)            AS graded_count,
            COUNT(*) - COUNT(grade) AS pending_count,
            ROUND(AVG(grade), 2)    AS avg_grade
     FROM takes""")

In [ ]:
# NOT IN 的隱藏地雷：名單裡混進一個 NULL，整句「全軍覆沒」
print("cid NOT IN ('C101','C999')      →", len(q("SELECT * FROM takes WHERE cid NOT IN ('C101','C999')")), f"列（期望 37）")
print("cid NOT IN ('C101', NULL)       →", len(q("SELECT * FROM takes WHERE cid NOT IN ('C101', NULL)")), "列 ← 一列都不剩！")
print()
print("原因：x NOT IN (a, NULL) ≡ x<>a AND x<>NULL；命中 a 時為 FALSE，未命中時為 UNKNOWN，都過不了 WHERE。")
print("最常中招的場景：NOT IN (SELECT ...) 而子查詢帶回了 NULL。")
print("防法：子查詢加 WHERE column_name IS NOT NULL。")

In [ ]:
# UNIQUE 遇上 NULL：沒填的 email 可以有很多個，填了的不准撞
con.execute("DROP TABLE IF EXISTS contacts")
con.execute("CREATE TABLE contacts(sid TEXT PRIMARY KEY NOT NULL, email TEXT UNIQUE)")
con.executemany("INSERT INTO contacts VALUES (?,?)",
                [("S001", "jr@stat.tw"), ("S002", None), ("S003", None)])   # 兩個 NULL 和平共處
con.commit()
print(q("SELECT * FROM contacts").to_string(index=False))
try:
    con.execute("INSERT INTO contacts VALUES ('S004', 'jr@stat.tw')")
except sqlite3.IntegrityError as e:
    print("\n✅ 重複 email 被擋 →", e)
print("→ UNIQUE 預設容許多個 NULL；這不表示 NULL = NULL 為 FALSE。必填仍要另外加 NOT NULL。")

In [ ]:
# NULL 的排序位置可以指定（SQLite 3.30+）：把「在修中」放最後
q("""SELECT sid, cid, grade FROM takes
     WHERE sid IN ('S001','S002')
     ORDER BY grade DESC NULLS LAST""")

# NULLS FIRST/LAST 是標準形式。SQLite 預設 ASC 把 NULL 放前面；PG 預設放後面。
# DESC 時兩者也相反；需要固定結果時明寫，不靠預設。

## 2.3 排序、去重、切頁

`LIMIT n OFFSET m` 是 SQLite／PG 共有擴充。標準形式是 `OFFSET m ROWS FETCH FIRST n ROWS ONLY`，PG 支援，但 SQLite 不支援此形式。分頁應用若要求穩定順序，要用 ORDER BY 並補上唯一鍵來打破同值。[PostgreSQL SELECT](https://www.postgresql.org/docs/18/sql-select.html)

In [ ]:
# 多鍵排序：先系所（升冪），同系再依年級（降冪）；LIMIT+OFFSET 是「分頁」的原型
q("SELECT name, dept, year FROM student ORDER BY dept, year DESC, sid LIMIT 8 OFFSET 0")

In [ ]:
q("SELECT DISTINCT dept FROM student ORDER BY dept")    # 有哪些系（去重）

## 2.4 運算式與內建函數

| 類別 | 常用款 |
|---|---|
| 字串 | `a \|\| b` 串接、`upper/lower`、`length`、`substr(s, start, length)`、`replace`、`trim`、`instr`、`printf` |
| 數值 | `round(x, digits)`、`abs`、`min(a,b)`／`max(a,b)`（雙參數版是逐列比！） |
| 條件 | `CASE WHEN … THEN … ELSE … END` |
| 空值 | `COALESCE(x, y, …)` 第一個非 NULL、`NULLIF(a, b)` 相等變 NULL（防除以零神器） |
| 型別 | `CAST(x AS REAL)`——**整數除法陷阱**的解藥 |
| 日期 | `date`／`datetime` 處理日期時間、`strftime` 擷取格式化欄位、`julianday` 計算日數 |

**方言對照**：CASE、COALESCE、NULLIF、CAST、`||` 等是標準形式；函數同名不保證參數與行為相同。

| 本講義寫法 | PostgreSQL／標準的差異 |
|---|---|
| `length`、`substr`、`replace` | PG 也有；標準代表形式另有 CHAR_LENGTH、SUBSTRING(... FROM ... FOR ...)；負的 substr 參數不可機械照搬 |
| `upper/lower` | SQLite 核心大小寫轉換主要限 ASCII；PG 依 locale／collation |
| `instr(s, needle)` | PG 用 strpos；標準形式 POSITION(needle IN s) |
| `printf` | SQLite 格式化函數；PG 的 format 規則不同，數字可用 to_char 或應用程式排版 |
| 多引數 `min(a,b)`／`max(a,b)` | SQLite 的逐列比較；PG 用 LEAST／GREATEST，但 NULL 行為不同 |
| `round(x,n)` | SQLite 的負 n 視為 0；PG 的兩參數形式要求 numeric，且負 n 能捨入十位、百位 |
| `CAST(x AS INTEGER)` | 語法標準，轉換規則不同；SQLite 可能把無效文字轉 0，PG 通常報錯 |

SQLite 的 grade REAL 可以直接 `ROUND(AVG(grade),2)`；PG 的 AVG(real) 回 double precision，需寫 `ROUND(CAST(AVG(grade) AS NUMERIC),2)` 或從 schema 使用適合的 NUMERIC。不要把名稱相同當成型別也相同。

依據：[SQLite 函數](https://www.sqlite.org/lang_corefunc.html)、[PG 字串函數](https://www.postgresql.org/docs/18/functions-string.html)、[PG 數值函數](https://www.postgresql.org/docs/18/functions-math.html)。

In [ ]:
# 字串函數＋整數除法陷阱（統計人算比例必踩）
print(q("SELECT sid || '－' || name AS display_label, length(name) AS name_length FROM student LIMIT 3").to_string(index=False))
print()
print("37 / 50   =", con.execute("SELECT 37 / 50").fetchone()[0], "  ← 整數除整數：小數被截掉！")
print("37 * 1.0 / 50 =", con.execute("SELECT 37 * 1.0 / 50").fetchone()[0])
print("CAST 寫法     =", con.execute("SELECT CAST(37 AS REAL) / 50").fetchone()[0])

In [ ]:
# 【選讀／工具箱】printf／substr 實戰：報表美化與遮罩（個資欄位顯示一半是應用系統日常）
print(q("""SELECT name,
             printf('NT$ %,d', CAST(salary AS INTEGER)) AS monthly_salary
          FROM instructor WHERE salary IS NOT NULL""").to_string(index=False))
print()
print(q("""SELECT name, substr(sid, 1, 2) || '**' AS masked_student_id FROM student LIMIT 3""").to_string(index=False))
# printf 的 %,d 千分位、%05d 補零、%.1f 小數——排版在 SQL 就能做掉一半

In [ ]:
# 【選讀／工具箱】字串清理實戰：把亂七八糟的電話格式洗乾淨（replace 連環拳——匯入舊資料的日常）
con.execute("DROP TABLE IF EXISTS phone_raw")
con.execute("CREATE TABLE phone_raw(who TEXT, phone TEXT)")
con.executemany("INSERT INTO phone_raw VALUES (?,?)", [
    ("佳蓉", "0912-345-678"), ("威廷", "0912 345 679"), ("雅筑", "(09)1234-5680"), ("孟軒", "0912345681")])
con.commit()
q("""SELECT who, phone,
        replace(replace(replace(replace(phone, '-', ''), ' ', ''), '(', ''), ')', '') AS cleaned,
        length(replace(replace(replace(replace(phone, '-', ''), ' ', ''), '(', ''), ')', '')) AS len_ok
     FROM phone_raw""")
# 洗完都是 10 碼 → 之後才能設 UNIQUE、才能比對。心法：先清洗、再約束——順序反了會被自己的髒資料卡死

## 2.5 日期時間專場（應用系統的核心技能）

**SQLite 用法**：本課以 TEXT 儲存日期。只有在**格式、補零、精度與時區一致**時，例如全用 `'2026-09-17'` 或全用同時區的 `'2026-09-17 14:30:00'`，字典序才能當作時間順序；不要混用日期、不同時區 offset 與多種時間分隔形式。

| 需求 | 寫法 |
|---|---|
| 台灣現在（本課單時區範例） | `datetime('now','+8 hours')`；今天 `date('now','+8 hours')` |
| 日期運算 | `date('now', '+8 hours', '+7 day')`、`date('now', '+8 hours', '-1 month')` |
| 對齊 | `date('now', '+8 hours', 'start of month')`、`date('now', '+8 hours', 'weekday 1')`（當天或之後第一個週一） |
| 取欄位／分組鍵 | `strftime('%Y-%m', d)` 年月、`'%w'` 星期幾（0=日）、`'%H'` 小時 |
| 差幾天 | `julianday(d2) - julianday(d1)` |

你的專題全都用得到：預約起迄、逾期天數、會籍效期、日結、報名截止⋯⋯

> **Colab 時區地雷**：SQLite 的 `'localtime'` 是「執行程式那台主機的當地時區」，不是使用者瀏覽器的時區；Colab host 通常是 UTC，所以 `localtime` 不會自動變台灣時間。本課全員在台灣的範例明寫 `'+8 hours'`。正式跨時區系統應在資料庫儲存 UTC（`datetime('now')`），顯示時再由應用層用 `zoneinfo.ZoneInfo`（如 `Asia/Taipei`）轉換；不要把所有使用者都假設在 UTC+8。

這組 date／datetime／strftime／julianday 與修飾詞屬 SQLite 日期介面。PG 通常使用 DATE／TIMESTAMP／INTERVAL 型別、CURRENT_DATE、EXTRACT、date_trunc 等；移植時需要重看型別、時區與月底規則。[SQLite 日期文件](https://www.sqlite.org/lang_datefunc.html)

In [ ]:
# 日期工具箱實測
date_demos = [
    ("今天",            "SELECT date('2026-09-17')"),
    ("7 天後（報名截止）","SELECT date('2026-09-17', '+7 day')"),
    ("本月第一天",       "SELECT date('2026-09-17', 'start of month')"),
    ("這是星期幾(0=日)", "SELECT strftime('%w', '2026-09-17')"),
    ("週一當天不前進", "SELECT date('2026-09-21', 'weekday 1')"),
    ("年月分組鍵",       "SELECT strftime('%Y-%m', '2026-09-17')"),
    ("跨年夜差幾天",     "SELECT julianday('2026-12-31') - julianday('2026-09-17')"),
]
for desc, sql in date_demos:
    print(f"{desc:12s} {sql[7:]:48s} → {con.execute(sql).fetchone()[0]}")

In [ ]:
# 【主線】月底陷阱：1/31 加一個月是幾月幾號？——日期運算的「正規化」行為要親眼看過
for desc, sql in [
    ("1/31 ＋ 1 個月",      "SELECT date('2026-01-31', '+1 month')"),               # 2 月沒有 31 → 溢到 3 月！
    ("本月月底的正解",       "SELECT date('2026-09-17', 'start of month', '+1 month', '-1 day')"),
    ("下月月底",            "SELECT date('2026-01-31', 'start of month', '+2 month', '-1 day')"),
]:
    print(f"{desc:14s} → {con.execute(sql).fetchone()[0]}")
print("\n→ '+1 month' 是「月份加一再正規化」：2026-02-31 不存在 → 自動變 2026-03-03。")
print("   月底結算可用 start of month 錨定再位移；固定每月 X 日或會籍到期，須先決定不存在該日時要截斷還是順延。")

In [ ]:
# 【選讀／工具箱】年齡與生日：日數換算只是近似；精確足歲要比「月-日」
# SQLite 的比較式結果是 0／1；精確足歲在生日尚未到時直接扣 1，跨資料庫時可明寫 CASE WHEN condition THEN 1 ELSE 0 END。
con.execute("DROP TABLE IF EXISTS bday")
con.execute("CREATE TABLE bday(name TEXT, birth TEXT)")
con.executemany("INSERT INTO bday VALUES (?,?)", [
    ("林佳蓉", "2005-03-14"), ("陳威廷", "2004-11-30"), ("吳孟軒", "2005-09-17")])
con.commit()
q("""SELECT name, birth,
        CAST((julianday('2026-09-17') - julianday(birth)) / 365.2425 AS INTEGER) AS approx_age,
        CAST(strftime('%Y','2026-09-17') AS INTEGER)
          - CAST(strftime('%Y', birth) AS INTEGER)
          - (strftime('%m-%d','2026-09-17') < strftime('%m-%d', birth)) AS age_years,
        CASE WHEN strftime('%m-%d', birth) = '09-17' THEN '🎂 今天生日！'
             WHEN strftime('%m-%d', birth) >  '09-17' THEN '今年還沒過'
             ELSE '今年過了' END AS birthday_status
     FROM bday""")

In [ ]:
# 應用場景演練：借閱到期與逾期天數（教師示範專題「圖書館系統」的核心查詢原型）
con.execute("DROP TABLE IF EXISTS loan_demo")
con.execute("CREATE TABLE loan_demo(who TEXT, book TEXT, due TEXT)")
con.executemany("INSERT INTO loan_demo VALUES (?,?,?)", [
    ("S001", "統計學習導論", "2026-09-10"),   # 已逾期
    ("S002", "資料庫概論",   "2026-09-20"),   # 快到期
    ("S003", "迴歸分析",     "2026-10-05"),   # 還早
])
con.commit()
q("""SELECT who, book, due,
        CAST(julianday('2026-09-17') - julianday(due) AS INTEGER) AS overdue_days,
        CASE WHEN due <  '2026-09-17' THEN '⚠️ 逾期'
             WHEN due <= date('2026-09-17', '+3 day') THEN '快到期'
             ELSE 'OK' END AS status
     FROM loan_demo ORDER BY due""")

In [ ]:
# CASE WHEN：成績 → 等第（資料重編碼 recode，統計日常）
q("""SELECT sid, cid, grade,
        CASE WHEN grade IS NULL THEN '在修'
             WHEN grade >= 90   THEN 'A'
             WHEN grade >= 80   THEN 'B'
             WHEN grade >= 70   THEN 'C'
             WHEN grade >= 60   THEN 'D'
             ELSE 'F' END AS letter_grade
     FROM takes ORDER BY grade DESC NULLS LAST LIMIT 10""")

In [ ]:
# COALESCE：顯示時把 NULL 換成人話（別改原始資料）；NULLIF：防除以零
print(q("SELECT sid, cid, COALESCE(CAST(grade AS TEXT), '（修課中）') AS grade FROM takes LIMIT 6")
        .to_string(index=False))
print()
print("除以零防護：SELECT 10.0 / NULLIF(0, 0) →",
      con.execute("SELECT 10.0 / NULLIF(0, 0)").fetchone()[0], "（NULL）")
print("SQLite 直接 10.0/0 也回 NULL：", con.execute("SELECT 10.0/0").fetchone()[0])
print("NULLIF 明確表達零分母變缺值的政策；在 PostgreSQL 可避免直接除零錯誤。")

## 2.6 無分組聚合：整張表濃縮成一列

`COUNT / SUM / AVG / MIN / MAX`——不加 `GROUP BY` 時，把符合條件的全部資料列視為一組。
一個超好用的統計技巧：**`AVG(CASE WHEN condition THEN 1.0 ELSE 0 END)` ＝ 條件成立的比例**。

**SQLite 用法**：比較結果可當作 `1`／`0`，所以 `SUM(grade < 60)` 能直接數不及格；遇到 NULL 則產生 NULL，`SUM` 會忽略它。這是 **SQLite 慣用簡寫，不是可攜 SQL**：PostgreSQL 等系統請用 `SUM(CASE WHEN … THEN 1 ELSE 0 END)`，或 `COUNT(*) FILTER (WHERE …)`。

兩種 SUM 寫法不能無條件視為等價：若非空輸入中的比較結果全是 NULL，`SUM(condition)` 回 NULL，而 `SUM(CASE WHEN condition THEN 1 ELSE 0 END)` 回 0。`COUNT(*) FILTER (WHERE condition)` 是 SQLite 3.30+／PG 都有的標準形式，空集合也回 0。要數人、算比例或保留未知，需先選定語意。

In [ ]:
# SQLite 布林簡寫 vs 標準 CASE：本資料答案相同，但要檢查全 NULL 情況
short_count = con.execute("SELECT SUM(grade < 60) FROM takes").fetchone()[0]
portable_count = con.execute("""SELECT SUM(CASE WHEN grade < 60 THEN 1 ELSE 0 END)
                                FROM takes""").fetchone()[0]
print("SUM(條件) =", short_count, "；SUM(CASE ...) =", portable_count)
assert short_count == portable_count

all_null = con.execute("""SELECT SUM(grade < 60),
    SUM(CASE WHEN grade < 60 THEN 1 ELSE 0 END),
    COUNT(*) FILTER (WHERE grade < 60)
    FROM (SELECT NULL AS grade)""").fetchone()
assert all_null == (None, 0, 0)
print("全 NULL 的 SUM(bool)、SUM(CASE)、COUNT FILTER：", all_null)

In [ ]:
q("""SELECT COUNT(*)                 AS enrollment_count,
            COUNT(DISTINCT sid)      AS student_count,
            ROUND(AVG(grade), 2)     AS avg_grade,
            MIN(grade) AS min_grade, MAX(grade) AS max_grade,
            ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3) AS pass_rate_including_pending
     FROM takes""")

In [ ]:
# 上面「及格率」分母是誰？NULL 列被 AVG 忽略了嗎？——沒有！CASE 把 NULL 變成 0 了，陷阱！
print("含在修（NULL 當不及格）：",
      con.execute("SELECT ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3) FROM takes").fetchone()[0])
print("只算有成績的（正解）　：",
      con.execute("""SELECT ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3)
                     FROM takes WHERE grade IS NOT NULL""").fetchone()[0])
print("→ 統計素養時刻：同一句「及格率」，分母定義不同，數字差一截。SQL 沒錯，是你要想清楚。")

In [ ]:
# SUM 的空集合陷阱與 TOTAL：報表右下角的「合計」欄防呆
print("SUM(grade)（沒有任何列中選時）  →", con.execute("SELECT SUM(grade) FROM takes WHERE grade > 200").fetchone()[0])
print("TOTAL(grade)（同條件）          →", con.execute("SELECT TOTAL(grade) FROM takes WHERE grade > 200").fetchone()[0])
print("COUNT(*)（同條件）              →", con.execute("SELECT COUNT(*) FROM takes WHERE grade > 200").fetchone()[0])
print()
print("→ SUM 對空集合回 NULL（會傳染給下游運算）；TOTAL 回 0.0。")
print("   通用形式是 COALESCE(SUM(x), 0)；TOTAL 是 SQLite 擴充且一律回浮點值，兩者型別不完全相同。")

In [ ]:
# 【選讀／工具箱】聚合不只算數字：group_concat 把一組值串成清單（報表「名單欄」神器）
print("所有系所：", con.execute("SELECT group_concat(DISTINCT dept) FROM student").fetchone()[0])
print("S001 修過：", con.execute("""SELECT group_concat(cid, '、')
                                    FROM (SELECT cid FROM takes
                                          WHERE sid = 'S001' ORDER BY cid)""").fetchone()[0])
print("→ 外層 ORDER BY 只排「聚合完的列」，不保證串接順序；這裡先在子查詢排好。")

# group_concat 是 SQLite／MySQL 常見的非標準函數；PG 使用 string_agg。
# 此排序子查詢是 SQLite 範例；跨引擎應依各自聚合函數的內部 ORDER BY 語法指定順序。

In [ ]:
# 【選讀／工具箱】應用：打工班表——「週末時數」佔比（strftime 重編碼 ＋ SUM(CASE) 合體）
con.execute("DROP TABLE IF EXISTS shift")
con.execute("CREATE TABLE shift(who TEXT, d TEXT, hrs REAL)")
con.executemany("INSERT INTO shift VALUES (?,?,?)", [
    ("佳蓉", "2026-09-14", 4), ("佳蓉", "2026-09-19", 6), ("佳蓉", "2026-09-20", 5),
    ("威廷", "2026-09-15", 8), ("威廷", "2026-09-19", 4),
    ("孟軒", "2026-09-16", 6), ("孟軒", "2026-09-17", 6)])
con.commit()
q("""SELECT SUM(hrs) AS total_hours,
        SUM(CASE WHEN strftime('%w', d) IN ('0','6') THEN hrs ELSE 0 END) AS weekend_hours,
        ROUND(SUM(CASE WHEN strftime('%w', d) IN ('0','6') THEN hrs ELSE 0 END) * 100.0
              / SUM(hrs), 1) AS weekend_pct
     FROM shift""")
# 這個查詢的統計單位是整張班表；週末與總時數採用相同資料範圍。

In [ ]:
# 簡單隨機抽樣的 SQL 寫法（做問卷抽獎、抽查都用得到）
q("SELECT sid, cid, grade FROM takes ORDER BY random() LIMIT 5")
# 注意：每次執行結果不同；要可重現的抽樣，可用 pandas 的 sample(random_state=...) 或預先存亂數欄

# random() 兩家都有但不是同一回傳型別：SQLite 是有正負的大整數；PG 無引數版是 [0,1) 浮點。

### 隨堂練習 C：觀念練習（想好再開）

**Q1.** `WHERE grade <> 100` 會不會回傳 grade 是 NULL 的列？
**Q2.** `COUNT(*)`、`COUNT(grade)`、`COUNT(DISTINCT grade)` 三者的大小關係？
**Q3.** UNIQUE 欄位可以有兩列都是 NULL 嗎？
**Q4.** 你寫 `WHERE cid NOT IN (SELECT ...)`，結果一列都不回。除了「真的沒有」，最可能的原因是？

<details><summary>答案</summary>

**A1.** 不會。`NULL <> 100` 是 UNKNOWN，被 WHERE 濾掉——「不等於」也帶不回 NULL。
**A2.** `COUNT(*) ≥ COUNT(grade) ≥ COUNT(DISTINCT grade)`（NULL 不算入後兩者；重複值再被 DISTINCT 壓縮）。
**A3.** 可以，這是 SQLite／PostgreSQL 的預設 UNIQUE 規則；`NULL = NULL` 仍是 UNKNOWN。必填需另加 NOT NULL。PG 也可用 NULLS NOT DISTINCT 改變唯一性規則。
**A4.** 子查詢帶回了 NULL：命中非 NULL 值時結果為 FALSE，未命中時為 UNKNOWN，兩者都過不了 WHERE。可先過濾子查詢的 NULL；若外層 x 本身也可為 NULL，仍需明確決定要如何處理。
</details>

### 隨堂練習 D：綜合一句

寫一句 SQL：「所有**有成績**的修課紀錄中，把成績四捨五入到十位（`ROUND(grade / 10.0, 0) * 10`），
統計每個值出現幾次」。

`GROUP BY` 把相同分桶結果歸成一組，`COUNT(*)` 計算各組筆數。這裡的 `GROUP BY bucket` 使用輸出別名，是 SQLite／PostgreSQL 都支援的寫法；也可重寫完整的分桶運算式。

<details><summary>答案</summary>

```sql
SELECT ROUND(grade / 10.0, 0) * 10 AS bucket, COUNT(*) AS n
FROM takes WHERE grade IS NOT NULL
GROUP BY bucket ORDER BY bucket;
```
對——單表函數玩到極限，自然就撞到「分堆統計」的需求。分組結果的每一列就是一個區間的筆數，也是**直方圖的資料底**。
</details>

**SQLite 注意**：ROUND 的負精度視為 0，不會捨入十位。`ROUND(88,-1)` 得 88.0，`ROUND(88/10.0,0)*10` 才是 90.0。[round 規則](https://www.sqlite.org/lang_corefunc.html#round)

In [ ]:
# 驗證十位捨入，避免只確認 SQL 能執行就以為答案正確
assert con.execute("SELECT ROUND(88,-1)").fetchone()[0] == 88.0
assert con.execute("SELECT ROUND(88/10.0,0)*10").fetchone()[0] == 90.0
buckets = q("""SELECT ROUND(grade / 10.0, 0) * 10 AS bucket, COUNT(*) AS n
    FROM takes WHERE grade IS NOT NULL GROUP BY bucket ORDER BY bucket""")
assert buckets.n.sum() == con.execute("SELECT COUNT(grade) FROM takes").fetchone()[0]
assert all(value % 10 == 0 for value in buckets.bucket)
print(buckets.to_string(index=False))

## 2.7【AI 協作】本單元示範：用中文描述換一張表

丟給 AI：

> 幫我寫 SQLite 的 CREATE TABLE：社團活動報名表。欄位：報名編號（自動編號主鍵）、
> 學號（必填、參照 student）、活動名稱（必填）、報名時間（預設現在）、
> 繳費金額（不可為負）。同一學號同一活動只能報名一次。

**驗收 SOP**（每次都做）：
1. 跑得過嗎？（貼進 Colab 執行）
2. 每條規則踩一腳：重複報名、負金額、幽靈學號——應該**全部**被擋（寫成 try/except 測試，像 1.2 那格）
3. 追問 AI 兩題，它答得出來、你也要答得出來：
   - 「`AUTOINCREMENT` 在 SQLite 可以拿掉嗎？為什麼？」
   - 「為什麼 `DEFAULT (datetime('now'))` 要加括號？」

> 專題共同要求第 8 條的「應該失敗的測試」，原型就是這套驗收 SOP。

Prompt 也應明寫：「SQL 的表名、欄名、別名與參數名一律英文；請標出 SQLite 專屬語法與最低版本，不要把 PostgreSQL 支援誤稱為 ANSI 標準。」

## 2.8【選做／加碼】VIEW：替查詢取一個可重用的名字

VIEW（檢視）是存在 schema 裡的 SELECT。它看起來像表，但一般 view **不另存一份查詢結果**；每次讀取時都從基底表重新計算，所以原表一改，view 看到的內容也跟著變。

    CREATE VIEW pending_takes_v AS
    SELECT sid, cid, semester
    FROM takes
    WHERE grade IS NULL;

之後可直接執行 <code>SELECT * FROM pending_takes_v</code>。它適合集中重複的欄位命名與篩選規則，但不能放執行時參數；會變動的條件仍寫在查 view 的 WHERE。SQLite 的 view 預設唯讀，新增或修改資料仍對基底表操作。

**標準 SQL**：CREATE VIEW 的基本形式。唯讀是這裡的 SQLite 行為，不能泛化到所有資料庫；PG 的部分簡單 view 可自動更新。SQLite 也可透過 INSTEAD OF trigger 定義如何把 view 的操作轉送到基底表。

In [ ]:
# 【選做／加碼】view 是「儲存的查詢」，不是結果快照
con.execute("DROP VIEW IF EXISTS pending_takes_v")
con.execute("""CREATE VIEW pending_takes_v AS
               SELECT sid, cid, semester
               FROM takes
               WHERE grade IS NULL""")
con.commit()

view_sql = con.execute(
    "SELECT sql FROM sqlite_master WHERE type='view' AND name='pending_takes_v'"
).fetchone()[0]
before_count = con.execute("SELECT COUNT(*) FROM pending_takes_v").fetchone()[0]
print("schema 裡存的是：", view_sql)
print("原本 view 有", before_count, "列")

target = con.execute(
    "SELECT sid, cid, semester FROM takes WHERE grade IS NULL LIMIT 1"
).fetchone()
con.execute("BEGIN")
con.execute(
    "UPDATE takes SET grade=80 WHERE sid=? AND cid=? AND semester=?",
    target,
)
after_count = con.execute("SELECT COUNT(*) FROM pending_takes_v").fetchone()[0]
print("替一筆在修紀錄填成績後，view 立刻變成", after_count, "列")
con.rollback()
restored_count = con.execute("SELECT COUNT(*) FROM pending_takes_v").fetchone()[0]
print("ROLLBACK 後回到", restored_count, "列")
q("SELECT * FROM pending_takes_v ORDER BY sid, cid LIMIT 5")

# 實作：單表查詢第 1–18 題

規則：每題一格，寫完跑出結果再開解答；**期望輸出**在註解裡幫你自我核對。
難度階梯：第 1–8 題是基本盤，第 9–12 題是主線綜合；**優先完成第 1–12 題**。
第 13–16 題是選做應用，第 17–18 題是加碼挑戰。

In [ ]:
# 第 1 題：統計系全部學生的 name、year，年級大的在前
# 期望：8 列，第一列是 4 年級




In [ ]:
# 第 2 題：學分數是 4 的課程（全部欄位）
# 期望：1 列（微積分）




In [ ]:
# 第 3 題：114-1 學期共有幾筆修課紀錄？（一個數字）
# 期望：18




In [ ]:
# 第 4 題：成績 >= 90 的 (sid, cid, grade)，高分在前
# 期望：11 列，最高 98




In [ ]:
# 第 5 題：目前「在修中」的修課紀錄 (sid, cid, semester)
# 期望：11 列




In [ ]:
# 第 6 題：學生來自哪些系所？（不重複）
# 期望：4 列




In [ ]:
# 第 7 題：課名含「統計」或「機率」的課
# 期望：2 列




In [ ]:
# 第 8 題：成績 70–79（含）的修課紀錄有幾筆？
# 期望：10




In [ ]:
# 第 9 題：列出 takes 的 sid, cid, grade，外加一欄 pass_status（是否及格）：
#          grade IS NULL → '在修'；>= 60 → '及格'；否則 '不及格'
# 期望：50 列；'不及格' 共 2 筆




In [ ]:
# 第 10 題：一句 SQL 回答——修課紀錄總數、有成績筆數、平均（2 位小數）
# 期望：50, 39, 81.08




In [ ]:
# 第 11 題：只看「有成績」的紀錄，及格率是多少？（3 位小數；用 AVG(CASE...) 技巧）
# 期望：0.949




In [ ]:
# 第 12 題：把學號的數字部分取出來轉成整數（substr + CAST），列出 sid 與該整數，取最大的 3 筆
# 期望：S020→20, S019→19, S018→18
# 變化版（做完的人）：改成列出「學號是偶數」的學生（提示：% 2）




In [ ]:
# 第 13 題（選做）：用 UPSERT 寫「借書證領取登記」——同一人重複登記不報錯、保留第一次時間
#   表：CREATE TABLE card_pickup(sid TEXT PRIMARY KEY NOT NULL, picked_at TEXT)
#   期望：對 S001 登記兩次後，表裡只有一筆、時間是第一次的




In [ ]:
# 第 14 題（選做）：把每門課輸出成一欄名牌：「C101｜統計學（一）｜3 學分」（printf 或 || 皆可）
# 期望：8 列，各一個字串




In [ ]:
# 第 15 題（選做）：只用一句 SELECT 比較：
#   (a) 純文字 '10' > '9'；(b) 兩邊明確 CAST 成 INTEGER 後再比較
# 欄名取為 text_order、numeric_order
# 期望：text_order=0, numeric_order=1




In [ ]:
# 第 16 題（選做）：從 raw_codes 找出「大寫 S 加三位 ASCII 數字」的合法代碼
# 期望：2 列，依序為 S001、S123
con.execute("DROP TABLE IF EXISTS raw_codes")
con.execute("CREATE TABLE raw_codes(code TEXT)")
con.executemany("INSERT INTO raw_codes VALUES (?)",
                [("S001",), ("S01",), ("s002",), ("S123",), ("S12A",), ("S0001",)])
con.commit()

# TODO：用 GLOB 寫 SELECT




In [ ]:
# 第 17 題（加碼挑戰）：建立 graded_takes_v view，只保留有成績的 sid, cid, semester, grade
# 接著查詢 view 的列數；寫成整格重跑也不報錯
# 期望：39




In [ ]:
# 第 18 題（加碼挑戰）：做一列修課儀表板
# 欄位：total_rows、graded_rows、pending_rows、passed_rows、failed_rows
# 提示：COUNT(*)、COUNT(grade)、SUM(CASE WHEN ...)
# 期望：50, 39, 11, 37, 2




<details><summary>📖 參考解答（先自己打完再開；跑得出期望輸出的寫法都算對）</summary>

```sql
-- 1
SELECT name, year FROM student WHERE dept = '統計' ORDER BY year DESC;
-- 2
SELECT * FROM course WHERE credits = 4;
-- 3
SELECT COUNT(*) FROM takes WHERE semester = '114-1';
-- 4
SELECT sid, cid, grade FROM takes WHERE grade >= 90 ORDER BY grade DESC;
-- 5
SELECT sid, cid, semester FROM takes WHERE grade IS NULL;
-- 6
SELECT DISTINCT dept FROM student ORDER BY dept;
-- 7
SELECT * FROM course WHERE title LIKE '%統計%' OR title LIKE '%機率%';
-- 8
SELECT COUNT(*) FROM takes WHERE grade BETWEEN 70 AND 79;
-- 9
SELECT sid, cid, grade,
       CASE WHEN grade IS NULL THEN '在修'
            WHEN grade >= 60   THEN '及格'
            ELSE '不及格' END AS pass_status
FROM takes;
-- 10
SELECT COUNT(*), COUNT(grade), ROUND(AVG(grade), 2) FROM takes;
-- 11
SELECT ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3)
FROM takes WHERE grade IS NOT NULL;
-- 12
SELECT sid, CAST(substr(sid, 2) AS INTEGER) AS n
FROM student ORDER BY n DESC LIMIT 3;
-- 12 變化版
SELECT sid, name FROM student WHERE CAST(substr(sid, 2) AS INTEGER) % 2 = 0;
-- 13（選做）
DROP TABLE IF EXISTS card_pickup;
CREATE TABLE card_pickup(sid TEXT PRIMARY KEY NOT NULL, picked_at TEXT);
INSERT INTO card_pickup VALUES ('S001', '2026-09-17 09:00:00')
ON CONFLICT(sid) DO NOTHING;
INSERT INTO card_pickup VALUES ('S001', '2026-09-17 10:00:00')
ON CONFLICT(sid) DO NOTHING;
SELECT * FROM card_pickup;
-- 14（選做）
SELECT printf('%s｜%s｜%d 學分', cid, title, credits) AS display_label FROM course;
-- 15（選做）
SELECT '10' > '9' AS text_order,
       CAST('10' AS INTEGER) > CAST('9' AS INTEGER) AS numeric_order;
-- 16（選做）
SELECT code FROM raw_codes
WHERE code GLOB 'S[0-9][0-9][0-9]'
ORDER BY code;
-- 17（加碼挑戰）
DROP VIEW IF EXISTS graded_takes_v;
CREATE VIEW graded_takes_v AS
SELECT sid, cid, semester, grade
FROM takes
WHERE grade IS NOT NULL;
SELECT COUNT(*) FROM graded_takes_v;
-- 18（加碼挑戰）
SELECT COUNT(*) AS total_rows,
       COUNT(grade) AS graded_rows,
       COUNT(*) - COUNT(grade) AS pending_rows,
       SUM(CASE WHEN grade >= 60 THEN 1 ELSE 0 END) AS passed_rows,
       SUM(CASE WHEN grade < 60 THEN 1 ELSE 0 END) AS failed_rows
FROM takes;
```
</details>

## 自主練習（非繳交）＆ 專題進度建議

**自主練習**（想多練的人，強烈建議）：自選一個有趣主題（追星、健身、遊戲、股票、社團⋯⋯），
1. 造 **2 張表**：一張有 FK 指向另一張，合計用滿 5 種以上約束；
2. 每張 `INSERT` ≥ 10 筆有意義的資料（讓 NULL 合理出現）；
3. 寫 **4 個「應該失敗」的踩點測試**（try/except，模仿 1.2 那格）；
4. 對它寫 **15 個查詢**：LIKE、BETWEEN／IN、`IS NULL`、多鍵 ORDER BY、CASE WHEN、日期函數、聚合、`AVG(CASE…)` 各至少一題。

**不用繳交**——可作為自行設計專題資料表的練習。

**專題進度建議**：把 Lab 核心第 1–12 題補完；有餘力再依序做第 13–18 題選做／加碼。檢查自己是否能將資料規則寫成約束，並解釋每個查詢的輸出。

---
## 附錄 A：可執行的 SQLite 語法參考

這段可在新的 SQLite 3.35+ 連線完整執行。PK／NOT NULL／CHECK／DEFAULT／FK 與基本增刪改查是通用概念；PRAGMA、自動 rowid、datetime 與 GLOB 是 SQLite 用法；UPSERT／RETURNING／LIMIT 是 SQLite／PG 共有擴充。

```sql
PRAGMA foreign_keys = ON;
CREATE TABLE cheat_parent(id INTEGER PRIMARY KEY);
INSERT INTO cheat_parent VALUES (1);
CREATE TABLE cheat_item(
  id INTEGER PRIMARY KEY,
  code TEXT NOT NULL UNIQUE,
  kind TEXT NOT NULL DEFAULT 'A' CHECK(kind IN ('A','B')),
  qty INTEGER NOT NULL DEFAULT 1 CHECK(qty > 0),
  unit_price INTEGER NOT NULL DEFAULT 10 CHECK(unit_price >= 0),
  amount INTEGER GENERATED ALWAYS AS (qty * unit_price),
  created_at TEXT NOT NULL DEFAULT (datetime('now','+8 hours')),
  parent_id INTEGER REFERENCES cheat_parent(id) ON DELETE CASCADE,
  start_date TEXT NOT NULL DEFAULT '2026-09-17',
  end_date TEXT NOT NULL DEFAULT '2026-09-18',
  CONSTRAINT valid_period CHECK(start_date < end_date)
);
ALTER TABLE cheat_item ADD COLUMN note TEXT;
INSERT INTO cheat_item(code, parent_id) VALUES ('S001', 1), ('S002', 1);
INSERT INTO cheat_item(code) VALUES ('S001') ON CONFLICT(code) DO NOTHING;
INSERT INTO cheat_item(code, qty) VALUES ('S001', 2)
ON CONFLICT(code) DO UPDATE SET qty = cheat_item.qty + excluded.qty;
INSERT INTO cheat_item(code) VALUES ('S003') RETURNING id, code;
UPDATE cheat_item SET kind='B' WHERE code='S002';
CREATE VIEW cheat_active AS SELECT code, amount FROM cheat_item WHERE kind='A';
SELECT code, amount FROM cheat_active ORDER BY amount DESC, code LIMIT 10;
SELECT code FROM cheat_item WHERE code GLOB 'S[0-9][0-9][0-9]' ORDER BY code;
SELECT COUNT(*) AS item_count, COALESCE(SUM(amount),0) AS total_amount FROM cheat_item;
DELETE FROM cheat_item WHERE code='S003';
DROP VIEW cheat_active;
DROP TABLE cheat_item;
DROP TABLE cheat_parent;
```

## 附錄 B：命令還能怎麼組合？

這是功能邊界的備查，不是新增的必做練習。跨引擎比較以 PostgreSQL 18 為例；先分清功能目的，再查該版本的語法。

| 命令 | 已有核心 | 其他能力與限制 |
|---|---|---|
| CREATE TABLE | 欄位、約束、default、generated | 兩家都有 TEMP TABLE（暫存表）、AS SELECT（用結果建表）；AS SELECT 不會完整複製原表約束與索引。SQLite 另有 STRICT、WITHOUT ROWID；PG 另有 identity、分割表、排除約束等 |
| INSERT | VALUES／SELECT／DEFAULT VALUES | 可組合 UPSERT、RETURNING。PG 可在 VALUES 某個位置寫 DEFAULT，SQLite 不支援；SQLite DEFAULT VALUES 也不能再接 UPSERT |
| UPDATE | WHERE 挑列、SET 計算一欄或多欄的新值 | 兩家都有子查詢、成組賦值、UPDATE FROM 與 RETURNING；PG 有 SET col=DEFAULT，SQLite 沒有 |
| UPDATE FROM | 用另一張表決定新值 | SQLite 3.33+ 與 PG 共有擴充；每個目標列應至多對到一筆來源，否則取哪筆不確定 |
| WITH 搭配寫入 | WITH 替查詢步驟取名字 | 兩家都可在 INSERT／UPDATE 前寫 WITH；PG 另可把 INSERT ... RETURNING 放進 WITH 當資料來源，SQLite 不支援 |
| 限制更新筆數 | 先明確定義目標列集合 | SQLite 的 UPDATE ORDER BY／LIMIT 需 SQLITE_ENABLE_UPDATE_DELETE_LIMIT 編譯選項；PG 沒有直接 UPDATE LIMIT |

`IF NOT EXISTS` 只避免名稱已存在的錯誤，不會驗證既有 schema 相符。標準另有 MERGE 命令，PG 有、SQLite 沒有；它與 UPSERT 的語意及併發保證不能直接畫等號。

官方完整語法：[SQLite CREATE TABLE](https://www.sqlite.org/lang_createtable.html)、[INSERT](https://www.sqlite.org/lang_insert.html)、[UPDATE](https://www.sqlite.org/lang_update.html)；[PostgreSQL CREATE TABLE](https://www.postgresql.org/docs/18/sql-createtable.html)、[INSERT](https://www.postgresql.org/docs/18/sql-insert.html)、[UPDATE](https://www.postgresql.org/docs/18/sql-update.html)。

### 錯誤家族速查
| 例外 | 意思 | 第一反應 |
|---|---|---|
| `OperationalError` | SQL 有問題 | 檢查表名／欄名／拼字 |
| `IntegrityError` | 資料違規 | 看訊息點名哪條約束 |
| `ProgrammingError` | API 用法錯 | 檢查 `?` 與參數 tuple（逗號！） |

## 附錄 C：讀物對照（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| 第 0 節 關聯模型與 key | §1.3、§2.1–2.3 | §2.1–2.2 |
| 1.1–1.7 DDL、約束、DML | §3.2、§4.4（integrity constraints） | §2.3 |
| 2.1–2.4 SELECT、WHERE、函數 | §3.3–3.4 | §6.1 |
| 2.2 NULL 三值邏輯 | §3.6 | §6.1.6–6.1.7 |
| 2.6 聚合 | §3.7 | §6.4 |

SQLite 官方文件（當字典用）：型別與 affinity https://sqlite.org/datatype3.html ・日期函數 https://sqlite.org/lang_datefunc.html ・UPSERT https://sqlite.org/lang_upsert.html